# Requirements and Libraries

In [1]:
pip install  lightgbm  lifelines xgboost catboost optuna fairlearn

Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install lifelines==0.29.0

In [3]:
# aquí dejo un "special import" (luego tengo temas con este import)
def _pip_install(pkg):
    try:
        __import__(pkg.split('==')[0].replace("-","_"))
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executeable, "-m", "pip", "install", pkg])
        
for pkg in ['holidays']:
    _pip_install(pkg)
        

In [4]:
#basics
import os
import sys
import re
import json
import warnings
import joblib
import unicodedata
import logging

#data manipulation
import pandas as pd
import numpy as np
import typing as t
import holidays

#Data Viz
import seaborn as sns
import matplotlib.pyplot as plt

#info
from pathlib import Path
from zoneinfo import ZoneInfo
from dataclasses import dataclass
from datetime import datetime, date

#google bigquery
from google.cloud import bigquery

#statistics
import scipy
from scipy import stats
from scipy.stats import ks_2samp
from lifelines import KaplanMeierFitter, CoxPHFitter
from math import pi
#sklearn
import sklearn
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.utils import Bunch
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.calibration import IsotonicRegression, calibration_curve
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

#modelos
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


#metrics
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss,confusion_matrix,classification_report
from fairlearn.metrics import MetricFrame


#optimization momdels
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import make_scorer

warnings.filterwarnings('ignore')
RNG_SEED = 42
np.random.seed(RNG_SEED)


# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

logger.info("✓ Librerías importadas y configuración completa")

2026-04-30 20:38:10,113 - INFO - ✓ Librerías importadas y configuración completa


# Data Load

In [5]:
client = bigquery.Client(project="spin-aip-singularity-comp-sb")

query = """ SELECT * FROM `spin-aip-singularity-comp-sb.model_activation.model_5trx_test_v31`"""

data = client.query(query).to_dataframe()



In [6]:
data.head()

,user_id,signup_date,signup_ts,userTypeIdentifier,channelUserIdentifier,accountLevel,stateName,gender,user_type,channelDetail,birth_date,birthState,Card_linked_date,IsActive,phn_confir,email_confir,phone_conf_ts,email_conf_ts,phn_confir_d7,email_confir_d7,both_confir_d7,premia_accountid,has_premia,activation_date_ever,activation_date_30d,label_activated_30d,tx_30d_count,tx_30d_amount,label_5tx_30d,first_tx_type,first_tx_amount,activation_channel,latest_tx_date,lifespan_days,days_since_last,tx_30d_from_activation,days_to_first_activation
0,48dac67d-6d0f-46bf-9a54-e4391d0cc2b6,2025-09-30,2025-09-30 17:55:51.348000+00:00,3,1,2,GR,female,HYBRID,POS,1986-03-07,GR,2025-09-30,True,1,1,2025-10-01 00:30:00.830000+00:00,2025-11-02 02:48:46.702000+00:00,1,0,0,ef1ed102-c459-49d4-9e4c-64456e5cf499,1,2025-09-30,2025-09-30,1,71,50788.1800,1,CASH_IN_AT_OXXO,2799.0000,CashIn_OXXO,2026-04-03,186,7,71,0
1,48db5e4a-72de-4d26-af0d-2d7472642c7a,2025-01-10,2025-01-11 00:12:39.970000+00:00,3,1,3,VE,male,HYBRID,POS,1997-06-25,TS,2025-01-10,True,1,1,2025-01-11 07:12:53.544000+00:00,2025-01-12 03:28:25.240000+00:00,1,1,1,ARI7HER,1,2025-01-10,2025-01-10,1,63,35950.8500,1,SPEI_CASH_IN,118.0000,SPEI/Transfer,2025-03-30,80,376,63,0
2,48db75c6-790c-4ec6-99ba-ae1c1e41ac0d,2026-02-21,2026-02-21 19:17:51.987000+00:00,3,1,2,EM,male,HYBRID,POS,1984-02-25,EM,2026-02-21,True,1,1,2026-02-24 10:31:16.986000+00:00,2026-03-30 10:37:56.500000+00:00,1,0,0,18a5c45f-7980-4e51-bcc0-9a702baae5da,1,2026-02-27,2026-02-27,1,32,15245.0500,1,SPEI_CASH_IN,3674.0000,SPEI/Transfer,2026-04-07,40,3,45,6
3,48db8749-aa3a-4f98-9707-ae8f9c233f35,2025-04-08,2025-04-09 03:07:05.465000+00:00,3,1,2,CS,male,HYBRID,POS,2006-06-04,CS,2025-04-08,True,1,0,2025-04-27 03:18:44.850000+00:00,NaT,0,0,0,3S7978Q,1,2025-04-25,2025-04-25,1,20,6276.0000,1,SPEI_CASH_IN,2198.0000,SPEI/Transfer,2025-05-24,30,321,50,17
4,48dbe6ae-7700-4b02-833e-7f1f4b8554a7,2025-09-10,2025-09-10 21:41:43.487000+00:00,3,1,2,EM,female,HYBRID,POS,2005-12-14,EM,2025-09-10,True,1,1,2025-09-13 04:36:37.964000+00:00,2025-09-13 04:46:32.844000+00:00,1,1,1,WDCRJIQ,1,2025-09-13,2025-09-13,1,36,9406.8100,1,CASH_IN_AT_OXXO,4000.0000,CashIn_OXXO,2026-04-09,209,1,40,3


In [7]:
df = data.copy(deep=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3834967 entries, 0 to 3834966
Data columns (total 37 columns):
 #   Column                    Dtype              
---  ------                    -----              
 0   user_id                   object             
 1   signup_date               object             
 2   signup_ts                 datetime64[us, UTC]
 3   userTypeIdentifier        Int64              
 4   channelUserIdentifier     Int64              
 5   accountLevel              Int64              
 6   stateName                 object             
 7   gender                    object             
 8   user_type                 object             
 9   channelDetail             object             
 10  birth_date                object             
 11  birthState                object             
 12  Card_linked_date          object             
 13  IsActive                  boolean            
 14  phn_confir                Int64              
 15  email_confir   

In [8]:
df.shape

(3834967, 37)

# Feature Engineering

In [9]:
@dataclass
class Config:
    project_id: str = "spin-aip-singularity-comp-sb"
    table_fqn: str = "spin-aip-singularity-comp-sb.model_activation.model_5trx_test_v31"
    label_col: str = "label_5tx_30d"
    signup_ts_col: str = "signup_ts"
    signup_date_col: str = "signup_date"
    tz_local: str = "America/Mexico_City"
    embargo_days: int = 3      
    holdout_days: int = 14 
    n_splits: int = 5
    train_sample_frac: float = 1.0
    activation_filter: int = 1
    random_state: int = RNG_SEED
    lift_fracs: t.Tuple[float, ...] = (0.01, 0.02, 0.05, 0.10)

CFG = Config()

In [10]:
# Leakage / IDs a ignorar en features
LEAKY_ALWAYS = {    
    #"label_activated_30d"
    "y_w0",
    "y_w1",
    "y_w7",
    "y_w30",
    "y_cum30",
    "label_5tx_30d",
    # info post-activación / post-window
    #"activation_date_ever",
    #"activation_date_30d",
    #"days_to_first_activation",
    #"tx_30d_count",
    #"tx_30d_amount",
    #"tx_30d_from_activation", 
    #"first_tx_type",
    #"first_tx_amount",
    #"latest_tx_date",
    # confirmaciones si no son estrictamente previas al cutoff de cada horizonte
#    "phn_confir","email_confir",
    #"phn_confir_d7",
    #"email_confir_d7",
    #"both_confir_d7",
    #'Card_linked_date',
    #"activation_*",
   # "*_30d_*",
    "*latest_tx*",
    #ID
    "user_id", 
    "userid",
    "channelUserIdentifier",
    "premia_accountid", 
    "accountid", 
    "member_id", 
    "spin_user_id", 
    "id"
}

In [11]:
LEAK_BAN = {
    #"activation_date_30d"
    "y_w0",
    "y_w1",
    "y_w7",
    "y_w30",
    "y_cum30",
    "label_activated_30d",
    "label_5tx_30d",
    #"activation_date_ever",
    #"days_to_first_activation",
    #"latest_tx_date",
    #"tx_30d_count",
    #"tx_30d_amount",
    #"tx_30d_from_activation",
    #"first_tx_type",
    #"first_tx_amount",
    #"activation_channel"
}

In [12]:
# Regex anti-leak (además del set LEAKY_ALWAYS existente)
LEAK_PATTERNS = [
    #r"(^|_)activation(_|$)", 
    #r"(^|_)first_tx(_|$)", 
    #r"(^|_)latest_tx(_|$)",
    #r"(^|_)tx_30d(_|$)", 
    #r_)days_to_first(_|$)", 
 #   r"(^|_)from_activation(_|$)"
]

In [13]:
def assert_no_regex_leak(df_like: pd.DataFrame):
    bad = []
    for c in df_like.columns:
        for pat in LEAK_PATTERNS:
            if re.search(pat, c, flags=re.IGNORECASE):
                bad.append(c); break
    assert len(bad) == 0, f"LEAKAGE by regex: quita columnas {sorted(set(bad))}"

def assert_no_labelish_cols(df_like):
    inter = [c for c in df_like.columns if c in LEAK_BAN]
    assert len(inter) == 0, f"LEAKAGE: quita columnas {inter}"


EXTRA_DROP_TS = {"phone_conf_ts", "email_conf_ts"}

# Normalización de estados (stateName -> siglas)
def _norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = re.sub(r"[^A-Z ]", "", s.upper())
    return re.sub(r"\s+", " ", s).strip()

STATE_TO_ABBR = {
    "AGUASCALIENTES":"AG","BAJA CALIFORNIA":"BC","BAJA CALIFORNIA SUR":"BS","CAMPECHE":"CM",
    "CHIAPAS":"CS","CHIHUAHUA":"CH","CIUDAD DE MEXICO":"DF","COAHUILA":"CO","COLIMA":"CL",
    "DURANGO":"DG","GUANAJUATO":"GT","GUERRERO":"GR","HIDALGO":"HG","JALISCO":"JA","MEXICO":"EM",
    "MICHOACAN":"MI","MORELOS":"MO","NAYARIT":"NA","NUEVO LEON":"NL","OAXACA":"OA","PUEBLA":"PU",
    "QUERETARO":"QT","QUINTANA ROO":"QR","SAN LUIS POTOSI":"SL","SINALOA":"SI","SONORA":"SO",
    "TABASCO":"TB","TAMAULIPAS":"TM","TLAXCALA":"TL","VERACRUZ":"VE","YUCATAN":"YU","ZACATECAS":"ZA"
}
STATE_SYNONYMS = {"CDMX":"CIUDAD DE MEXICO","ESTADO DE MEXICO":"MEXICO","EDOMEX":"MEXICO"}

# birthState canon + buckets regionales
CANON = {
    "SR":"SO","SO":"SO","VZ":"VE","VE":"VE","YN":"YU","YU":"YU","JC":"JA","JA":"JA","MC":"MI","MI":"MI",
    "TS":"TM","TM":"TM","TC":"TB","TB":"TB","CC":"CL","CL":"CL","DF":"DF","EM":"EM","NL":"NL","BC":"BC",
    "BS":"BS","SI":"SI","NA":"NA","DG":"DG","ZA":"ZA","AG":"AG","SL":"SL","HG":"HG","MO":"MO","TL":"TL",
    "PU":"PU","QT":"QT","GT":"GT","OA":"OA","CM":"CM","CS":"CS","CO":"CO","GR":"GR","QR":"QR","CH":"CH",
    "MS":"MI","MN":"MI","SP":"SL","NE":"NL","OC":"OA","PL":"PU","NT":"NA","ZS":"ZA","AS":"AG","UN":"OT", None:"OT"
}
REGION_BUCKET = {
    "BC":1,"SO":1,"CH":1,"CO":1,"NL":1,"TM":1,                # Norte
    "BS":2,"SI":2,"NA":2,"DG":2,"ZA":2,                       # Norte-Occidente
    "JA":3,"AG":3,"CL":3,"MI":3,"SL":3,                       # Centro-Norte
    "DF":4,"EM":4,"HG":4,"MO":4,"TL":4,"PU":4,"QT":4,"GT":4,  # Centro-País
    "CS":5,"TB":5,"CM":5,"YU":5,"QR":5,"OA":5,"GR":5,"VE":5,  # Sur-Sureste
    "OT":0
}

# Mapeos categóricos
GENDER_MAP = {"female":1, "male":0}
USER_TYPE_MAP = {"HYBRID":0, "DIGITAL":1, "ANALOG":2}
CHANNEL_DETAIL_MAP = {"ORGANIC":0,"COLLABORATOR":1,"POS":2,"SPIN_PREMIA":3,"DIGITAL_ORGANIC":4,"DIGITAL":5}


CASH_IN_TX = {
        "CASH_IN_AT_OXXO",
        "CASH_IN_AT_OXXO_QR",
        "CASH_OUT_WITH_CARD_AT_OXXO", 
        "CASH_OUT_AT_OXXO",
        "CASH_OUT_AT_MERCHANT",
        "CARD_PURCHASE",
        "CARD_ATM_WITHDRAWAL"
}
SPEI_TX = {
        "SPEI_CASH_IN",
        "TRANSFER_TO_CARD",
        "TRANSFER_TO_CLABE",
        "P2P_TRANSFER_TARGET",
        "P2P_TRANSFER_TARGET_CLABE",
        "P2P_TRANSFER_TARGET",
        "P2P_TRANSFER_TARGET_CARD"}

P2P_TX ={
        "P2P_TRANSFER_TARGET_CLABE",
         "P2P_TRANSFER_TARGET",
         "P2P_TRANSFER_TARGET_CARD",
         "P2P_TRANSFER_SOURCE_CARD",
         "P2P_TRANSFER_SOURCE_CLABE",
         "P2P_TRANSFER_SOURCE",
         "IN_APP_PURCHASE_TAE",
         "IN_APP_PURCHASE_BILLPAYMENT",
         "QR_MERCHANT_PAYMENT",
         "GIFT_CARD_PURCHASE",
         "INTERNATIONAL_REMITTANCE_CASH_IN"}

TX_TYPE_MAP = {
    **{k: 0 for k in CASH_IN_TX},  # 0 = Cash_In
    **{k: 1 for k in SPEI_TX},     # 1 = SPEI
    **{k: 2 for k in P2P_TX}       # 2 = P2P
}

In [14]:
class FeatureBuilder:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.ohe_states_: t.List[str] = []
        # Pipeline de normalización
        self.scaler_: RobustScaler | None = None
        self.numeric_cols_: t.List[str] = []
        self._fitted_scaler: bool = False

    @staticmethod
    def _state_to_abbr(s: t.Any) -> str:
        if pd.isna(s): return "OT"
        s = str(s)
        if s.upper() in REGION_BUCKET: return s.upper()
        s2 = _norm(s)
        s2 = STATE_SYNONYMS.get(s2, s2)
        return STATE_TO_ABBR.get(s2, "OT")

    @staticmethod
    def _canon_birthstate(s: t.Any) -> str:
        if pd.isna(s): return "OT"
        s = str(s).upper()
        return CANON.get(s, s if s in REGION_BUCKET else "OT")

    def _mk_time_feats(self, df: pd.DataFrame) -> pd.DataFrame:
        ts = pd.to_datetime(df[self.cfg.signup_ts_col], utc=True).dt.tz_convert(ZoneInfo(self.cfg.tz_local))
        df = df.copy()
        df["signup_dow"] = ts.dt.weekday.astype("int16")
        df["signup_week"] = ts.dt.isocalendar().week.astype("int16")
        df["signup_month"] = ts.dt.month.astype("int16")
        hr = ts.dt.hour
        df["signup_daypart"] = np.select([(hr>=5)&(hr<=11),(hr>=12)&(hr<=17)],[0,1],default=2).astype("int8")
        years = list({d.year for d in pd.to_datetime(df[self.cfg.signup_date_col]).dt.date})
        mx_hol = holidays.MX(years=years)
        dates = pd.to_datetime(df[self.cfg.signup_date_col]).dt.date
        df["is_holiday_mx"] = dates.map(lambda d: 1 if d in mx_hol else 0).astype("int8")
        day = ts.dt.day
        eom = (ts + pd.offsets.MonthEnd(0)).dt.day
        df["near_payday_any"] = ((np.abs(day-1)<=3)|(np.abs(day-15)<=3)|(np.abs(day-eom)<=3)).astype("int8")
        df["near_payday_1st"] = (np.abs(day-1)<=3).astype("int8")
        df["near_payday_15"]  = (np.abs(day-15)<=3).astype("int8")
        df["near_payday_eom"] = (np.abs(day-eom)<=3).astype("int8")
        return df

    def fit(self, df: pd.DataFrame):
        st = df["stateName"].map(self._state_to_abbr)
        self.ohe_states_ = sorted(st.dropna().unique().tolist())
        if "OT" not in self.ohe_states_: self.ohe_states_.append("OT")
        return self

    def transform(self, df: pd.DataFrame) -> Bunch:
        df = df.copy()
        s_ts = pd.to_datetime(df[CFG.signup_ts_col], utc=True)

        # Categóricas core
        df["gender_bin"] = df["gender"].map(GENDER_MAP).astype("float32")
        df["user_type_tri"] = df["user_type"].map(USER_TYPE_MAP).astype("float32")
        df["channel_detail_code"] = df["channelDetail"].map(CHANNEL_DETAIL_MAP).astype("float32")

        # birth bucket + edad
        bcanon = df["birthState"].map(self._canon_birthstate)
        df["birth_bucket"] = bcanon.map(REGION_BUCKET).astype("float32")
        bdate = pd.to_datetime(df["birth_date"], errors="coerce", utc=True)
        df["age_years"] = ((s_ts - bdate).dt.days/365.25).astype("float32")

        # Time features
        df = self._mk_time_feats(df)

        # state OHE
        st = df["stateName"].map(self._state_to_abbr)
        for ab in self.ohe_states_:
            df[f"state_{ab}"] = (st==ab).astype("int8")

        # Confirmaciones / flags: Int nulos a 0
        if "phone_conf_ts" in df.columns:
            phn_ts = pd.to_datetime(df["phone_conf_ts"], errors="coerce", utc=True)
            df["phn_confir"] = (phn_ts < s_ts).fillna(False).astype("int8")
        else:
            df["phn_confir"] = 0

        if "email_conf_ts" in df.columns:
            email_ts = pd.to_datetime(df["email_conf_ts"], errors="coerce", utc=True)
            df["email_confir"] = (email_ts < s_ts).fillna(False).astype("int8")
        else:
            df["email_confir"] = 0

        # Card_linked_date -> deltas sin fuga
        if "Card_linked_date" in df.columns:
            card_dt = pd.to_datetime(df["Card_linked_date"], errors="coerce", utc=True)
            # Cambiar <= por < (estrictamente antes)
            before = card_dt < s_ts  
            lag_days = (s_ts - card_dt).dt.days.astype("float32")
            df["card_linked_before_signup"] = before.fillna(False).astype("int8")
            # Solo crear lag_days si ocurrió ANTES (no <=)
            df["card_linked_lag_days"] = np.where(before, lag_days, np.nan).astype("float32")
            df = df.drop(columns=["Card_linked_date"])

        LEAKY_FEATURES = {
            #'lifespan_days',        
            #'days_since_last',
            'tx_30d_count',
            'tx_30d_from_activation',
            #'p_activation_30d', 
            'p_tx_0', 
            'p_tx_1', 
            'p_tx_2'
        }
            
        # Armar X
        drop_cols = set(LEAKY_ALWAYS) | {
            "stateName","gender","user_type","channelDetail","birthState","birth_date",
            CFG.signup_date_col, CFG.signup_ts_col
        } | set(EXTRA_DROP_TS) | LEAKY_FEATURES  

        drop_cols = [c for c in drop_cols if c in df.columns]
        X = df.drop(columns=drop_cols, errors="ignore")

        # Limpiar tipos
        # 1) Si queda algún dtype extension de BigQuery -> fuera
        bad_ext = [c for c in X.columns if "db_dtypes" in str(X[c].dtype).lower() or "dbdate" in str(X[c].dtype).lower()]
        X = X.drop(columns=bad_ext, errors="ignore")

        # 2) Casts seguros
        for c in X.columns:
            if pd.api.types.is_integer_dtype(X[c]) or str(X[c].dtype).startswith("Int"):
                X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0).astype("float32")
            elif pd.api.types.is_float_dtype(X[c]):
                X[c] = X[c].astype("float32")
            elif pd.api.types.is_bool_dtype(X[c]):
                X[c] = X[c].astype("int8")
            elif pd.api.types.is_datetime64_any_dtype(X[c]):
                X = X.drop(columns=[c])

        # 3) Objetos -> fuera
        obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
        if obj_cols:
            X = X.drop(columns=obj_cols)

        # Normalización con RobustScaler
        if not self._fitted_scaler:
            # Primera vez: identificar columnas numéricas y fit scaler
            numeric_cols = [c for c in X.columns
                            if not c.startswith('state_')  # Excluir one-hot de estados
                            and X[c].dtype == 'float32'     # Solo float32
                            and X[c].nunique() > 10]        # Excluir binarias
            
            self.numeric_cols_ = numeric_cols
            
            if len(numeric_cols) > 0:
                self.scaler_ = RobustScaler()
                self.scaler_.fit(X[numeric_cols])
                self._fitted_scaler = True
                print(f'RobustScaler fitted con {len(numeric_cols)} features numéricas')
            else:
                print('No se encontraron features numéricas para normalizar')
                self._fitted_scaler = True
        
        # Aplicar normalización si existe
        if self.scaler_ is not None and len(self.numeric_cols_) > 0:
            existing_cols = [c for c in self.numeric_cols_ if c in X.columns]
            if len(existing_cols) > 0:
                X[existing_cols] = self.scaler_.transform(X[existing_cols])

        # y binaria
        y = pd.to_numeric(df[CFG.label_col], errors="coerce").fillna(0).astype(int).values
        
        # Verificación final de leakage
        assert_no_regex_leak(X)
        LEAK_BAN = LEAKY_ALWAYS  | {"label_5tx_30d"}
        #assert_no_labelish_cols(X)
        
        meta = pd.DataFrame({
            "user_id": df.get("user_id", pd.Series(index=df.index, dtype="object")),
            "signup_date": pd.to_datetime(df[CFG.signup_date_col], errors="coerce"),
            "gender": df["gender"].astype(str),
            "channelDetail": df["channelDetail"].astype(str),
            "state_abbr": st.astype(str)
        })
        return Bunch(X=X, y=y, meta=meta)

In [15]:
df_5tx = df[df["label_activated_30d"]==1].reset_index(drop=True)


print(f"Threshold de Activación aplicado: {CFG.activation_filter}")
print("Shape df_5tx (activados y high-score):", df_5tx.shape)
print("Rate 5tx en esta población:", df_5tx["label_5tx_30d"].mean())

df_5tx
# Construcción de features con FeatureBuilder
fb = FeatureBuilder(CFG).fit(df_5tx)
bunch = fb.transform(df_5tx)

X_all = bunch.X.values.astype("float32") 
feature_names = list(bunch.X.columns)  # Guardar nombres para después

#X_all = bunch.X.astype("float32")       
y_all = df_5tx["label_5tx_30d"].astype(int).values  

signup_dates = pd.to_datetime(df_5tx[CFG.signup_date_col], errors="coerce")

print("X_all shape:", X_all.shape)
print("y_all rate (5tx):", y_all.mean())

Threshold de Activación aplicado: 1
Shape df_5tx (activados y high-score): (2371692, 37)
Rate 5tx en esta población: 0.6868826137626639
RobustScaler fitted con 9 features numéricas
X_all shape: (2371692, 64)
y_all rate (5tx): 0.6868826137626639


In [16]:
feature_names

['userTypeIdentifier',
 'accountLevel',
 'IsActive',
 'phn_confir',
 'email_confir',
 'phn_confir_d7',
 'email_confir_d7',
 'both_confir_d7',
 'has_premia',
 'label_activated_30d',
 'tx_30d_amount',
 'first_tx_amount',
 'lifespan_days',
 'days_since_last',
 'days_to_first_activation',
 'gender_bin',
 'user_type_tri',
 'channel_detail_code',
 'birth_bucket',
 'age_years',
 'signup_dow',
 'signup_week',
 'signup_month',
 'signup_daypart',
 'is_holiday_mx',
 'near_payday_any',
 'near_payday_1st',
 'near_payday_15',
 'near_payday_eom',
 'state_AG',
 'state_BC',
 'state_BS',
 'state_CH',
 'state_CL',
 'state_CM',
 'state_CO',
 'state_CS',
 'state_DF',
 'state_DG',
 'state_EM',
 'state_GR',
 'state_GT',
 'state_HG',
 'state_JA',
 'state_MI',
 'state_MO',
 'state_NA',
 'state_NL',
 'state_OA',
 'state_OT',
 'state_PU',
 'state_QR',
 'state_QT',
 'state_SI',
 'state_SL',
 'state_SO',
 'state_TB',
 'state_TL',
 'state_TM',
 'state_VE',
 'state_YU',
 'state_ZA',
 'card_linked_before_signup',
 'c

In [17]:
# MATRIZ DE CORRELACIÓN PEARSON + SPEARMAN CONTRA EL TARGET

# Convertimos X a DataFrame con nombres de features
if hasattr(bunch.X, "columns"):
    X_df = bunch.X.copy()
    cols = list(X_df.columns)
else:
    cols = getattr(bunch, "feature_names", [f"f_{i}" for i in range(bunch.X.shape[1])])
    X_df = pd.DataFrame(bunch.X, columns=cols)

# Target
y = df_5tx["label_5tx_30d"].astype(int)

# PEARSON
pearson_corr = X_df.apply(lambda col: col.corr(y, method="pearson"))

# SPEARMAN
spearman_corr = X_df.apply(lambda col: col.corr(y, method="spearman"))

# Compactamos en un solo DataFrame
corr_df = pd.DataFrame({
    "feature": cols,
    "pearson": pearson_corr.values,
    "spearman": spearman_corr.values
})

# Ordenar por la correlación absoluta más alta
corr_df["abs_pearson"] = corr_df["pearson"].abs()
corr_df["abs_spearman"] = corr_df["spearman"].abs()

corr_df_sorted = corr_df.sort_values(by=["abs_pearson", "abs_spearman"], ascending=False)

print("Top 20 features más correlacionadas con label_5tx_30d:")
display(corr_df_sorted.head(20))

# FLAGS DE ALERTA
THRESHOLD_ALERT = 0.60  # ajusta según el nivel de paranoia

suspicious = corr_df_sorted[
    (corr_df_sorted["abs_pearson"] > THRESHOLD_ALERT) |
    (corr_df_sorted["abs_spearman"] > THRESHOLD_ALERT)
]

if suspicious.shape[0] > 0:
    print("\n⚠️ ALERTA: Features con correlación ALTA con el target")
    display(suspicious)
else:
    print("\n✔️ No se detectaron features con correlación fuerte (Pearson o Spearman) contra el target.")


Top 20 features más correlacionadas con label_5tx_30d:


,feature,pearson,spearman,abs_pearson,abs_spearman
10,tx_30d_amount,0.2535,0.5537,0.2535,0.5537
13,days_since_last,-0.2436,-0.2913,0.2436,0.2913
14,days_to_first_activation,-0.2223,-0.1213,0.2223,0.1213
0,userTypeIdentifier,0.2208,0.2159,0.2208,0.2159
16,user_type_tri,-0.2208,-0.2159,0.2208,0.2159
12,lifespan_days,0.2198,0.2806,0.2198,0.2806
5,phn_confir_d7,0.1781,0.1781,0.1781,0.1781
19,age_years,-0.1490,-0.1388,0.1490,0.1388
1,accountLevel,0.1210,0.1221,0.1210,0.1221
7,both_confir_d7,0.1120,0.1120,0.1120,0.1120



✔️ No se detectaron features con correlación fuerte (Pearson o Spearman) contra el target.


# Models

In [18]:
# Orden cronológico y definición de train / embargo / holdout 
order = np.argsort(signup_dates.values)
signup_dates = signup_dates.iloc[order].reset_index(drop=True)
X_all = X_all[order]
y_all = y_all[order]

max_date = signup_dates.max()
holdout_start = max_date - pd.Timedelta(days=CFG.holdout_days)
embargo_end = holdout_start
train_end = holdout_start - pd.Timedelta(days=CFG.embargo_days)

train_mask = signup_dates <= train_end
holdout_mask = signup_dates > holdout_start
embargo_mask = (~train_mask) & (~holdout_mask)

print("Última fecha:", max_date.date())
print("Train hasta:", train_end.date())
print("Embargo entre:", train_end.date(), "y", holdout_start.date())
print("Holdout desde:", holdout_start.date())
print("N train   :", train_mask.sum())
print("N embargo :", embargo_mask.sum())
print("N holdout :", holdout_mask.sum())


Última fecha: 2026-04-08
Train hasta: 2026-03-22
Embargo entre: 2026-03-22 y 2026-03-25
Holdout desde: 2026-03-25
N train   : 2291919
N embargo : 16586
N holdout : 63187


In [19]:
X_train_full = X_all[train_mask.values]
y_train_full = y_all[train_mask.values]

n_train = X_train_full.shape[0]
split_idx = int(n_train * 0.8)

X_tr, X_val = X_train_full[:split_idx], X_train_full[split_idx:]
y_tr, y_val = y_train_full[:split_idx], y_train_full[split_idx:]

print("Train interno:", X_tr.shape[0], "obs")
print("Valid interno:", X_val.shape[0], "obs")


Train interno: 1833535 obs
Valid interno: 458384 obs


In [20]:
# Modelo
model_5tx = LGBMClassifier(
    n_estimators=400,
    max_depth=8,
    learning_rate=0.05,
    class_weight="balanced",
    n_jobs=-1,
    random_state=CFG.random_state,
    deterministic=True,
    force_col_wise=True,
    verbose=-1
)

model_5tx.fit(X_tr, y_tr)

p_val = model_5tx.predict_proba(X_val)[:, 1]

print("AP (valid):", average_precision_score(y_val, p_val))
print("AUC (valid):", roc_auc_score(y_val, p_val))
print("Brier (valid):", brier_score_loss(y_val, p_val))

AP (valid): 0.9802187913917287
AUC (valid): 0.9574866447717287
Brier (valid): 0.08302625362220155


In [21]:
def objective_temporal_single(trial, model_name, X_tr, y_tr, X_val, y_val):
    """Objective con un solo split temporal train/val."""
    model = build_model(model_name, trial)

    model.fit(X_tr, y_tr)
    p_val = model.predict_proba(X_val)[:, 1]

    ap = average_precision_score(y_val, p_val)
    return ap

In [22]:
def lift_at_fracs(y_true: np.ndarray, y_score: np.ndarray, fracs) -> dict:
    """Devuelve capture_rate y lift para cada fracción."""
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    total_pos = y_true.sum()

    res = {}
    for f in fracs:
        k = max(1, int(len(y_true) * f))
        captured = y_sorted[:k].sum()
        if total_pos > 0:
            capture_rate = captured / total_pos
            lift = capture_rate / f
        else:
            capture_rate, lift = np.nan, np.nan
        res[f] = {
            "capture_rate": float(capture_rate),
            "lift": float(lift),
            "n_users": int(k),
        }
    return res

In [23]:
#  Evaluación en HOLDOUT 
X_holdout = X_all[holdout_mask.values]
y_holdout = y_all[holdout_mask.values]

p_holdout = model_5tx.predict_proba(X_holdout)[:, 1]

metrics_val = {
    "AP": float(average_precision_score(y_val, p_val)),
    "AUC": float(roc_auc_score(y_val, p_val)),
    "Brier": float(brier_score_loss(y_val, p_val)),
    "Lift": lift_at_fracs(y_val, p_val, CFG.lift_fracs),
}

metrics_holdout = {
    "AP": float(average_precision_score(y_holdout, p_holdout)),
    "AUC": float(roc_auc_score(y_holdout, p_holdout)),
    "Brier": float(brier_score_loss(y_holdout, p_holdout)),
    "Lift": lift_at_fracs(y_holdout, p_holdout, CFG.lift_fracs),
}

print("\n MÉTRICAS 5TX (VALID) ")
print(json.dumps(metrics_val, indent=2))
print("\n MÉTRICAS 5TX (HOLDOUT) ")
print(json.dumps(metrics_holdout, indent=2))


 MÉTRICAS 5TX (VALID) 
{
  "AP": 0.9802187913917287,
  "AUC": 0.9574866447717287,
  "Brier": 0.08302625362220155,
  "Lift": {
    "0.01": {
      "capture_rate": 0.014362223872692346,
      "lift": 1.4362223872692346,
      "n_users": 4583
    },
    "0.02": {
      "capture_rate": 0.02872758154941539,
      "lift": 1.4363790774707694,
      "n_users": 9167
    },
    "0.05": {
      "capture_rate": 0.07182365457958452,
      "lift": 1.4364730915916903,
      "n_users": 22919
    },
    "0.1": {
      "capture_rate": 0.14362850633498483,
      "lift": 1.4362850633498483,
      "n_users": 45838
    }
  }
}

 MÉTRICAS 5TX (HOLDOUT) 
{
  "AP": 0.9165421081336995,
  "AUC": 0.9363026867690124,
  "Brier": 0.11122466072728812,
  "Lift": {
    "0.01": {
      "capture_rate": 0.023263530452735586,
      "lift": 2.3263530452735584,
      "n_users": 631
    },
    "0.02": {
      "capture_rate": 0.04652706090547117,
      "lift": 2.3263530452735584,
      "n_users": 1263
    },
    "0.05": {
   

In [24]:

# Predecir la probabilidad condicional para TODOS los usuarios activados

p_cond_all = model_5tx.predict_proba(X_all)[:, 1]

# 2. Crear el DataFrame de scores copiando el original ordenado
df_5tx_scores = df_5tx.iloc[order].copy()

# 3. Asignar la probabilidad condicional  
df_5tx_scores["p_5tx_30d"] = p_cond_all


# Guardar resultados (Artefacto de Producción)
cols_export = ["user_id", "signup_date", "p_5tx_30d"]
df_export = df_5tx_scores[cols_export]


In [25]:
df_export

,user_id,signup_date,p_5tx_30d
366883,fc38a440-1366-4b91-880b-9e34199f1129,2025-01-01,0.0659
770019,608860b6-7c99-4571-a5fb-5faa11fd493c,2025-01-01,0.2239
493342,fadeae5f-3323-400c-bdd3-a4345acc6948,2025-01-01,0.0141
904234,873f4303-9194-419c-a9b6-814aa26cab4d,2025-01-01,0.2271
1599296,611f9e93-5213-45bc-9c46-dc3e2699e960,2025-01-01,0.9214
...,...,...,...
1148167,6230b4ce-1062-4068-a32f-dc27e3ab6298,2026-04-08,0.1783
498060,f47b3ac1-36bb-4fdd-acb1-ec7abe625d74,2026-04-08,0.0098
1419472,863d46d5-3a9c-4f3a-8c98-104053716e73,2026-04-08,0.3535
862246,fc4f3bd8-54cc-4233-bc49-2b519ff6adbe,2026-04-08,0.1435


In [26]:
from pathlib import Path
from datetime import datetime


artifacts_dir = Path("mlops_artifacts_v2")
artifacts_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
export_path = artifacts_dir / f"df_export_5tx_30d_{ts}.csv"

df_export.to_csv(export_path, index=False, encoding="utf-8")

print(f"✅ CSV guardado en: {export_path.resolve()}")
print(f"Rows: {len(df_export):,} | Cols: {df_export.shape[1]}")


✅ CSV guardado en: /home/jupyter/5_Trx model/mlops_artifacts_v2/df_export_5tx_30d_2026-04-30_20:40:09.csv
Rows: 2,371,692 | Cols: 3


# Artifacts

In [27]:
from pathlib import Path
import json, sys, platform, cloudpickle
from datetime import datetime, timezone

In [28]:
ARTIFACTS_DIR = Path("mlops_artifacts_v2")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [29]:
MODEL_PATH = ARTIFACTS_DIR / "model_5tx.pkl"
FB_PATH = ARTIFACTS_DIR / "feature_builder.pkl"
CFG_PATH = ARTIFACTS_DIR / "cfg.pkl"
SCHEMA_PATH = ARTIFACTS_DIR / "feature_schema.json"

In [30]:

#guardar modelo 
with open(MODEL_PATH, "wb") as f:
    cloudpickle.dump(model_5tx, f)  
print(f" Modelo guardado: {MODEL_PATH} ({MODEL_PATH.stat().st_size:,} bytes)")

# 2. GUARDAR FeatureBuilder
with open(FB_PATH, "wb") as f:
    cloudpickle.dump(fb, f)
print(f"FeatureBuilder guardado: {FB_PATH} ({FB_PATH.stat().st_size:,} bytes)")

# 3. GUARDAR Config
with open(CFG_PATH, "wb") as f:
    cloudpickle.dump(CFG, f)
print(f" Config guardada: {CFG_PATH}")

# 4. GUARDAR Schema

feature_schema = list(bunch.X.columns)  
SCHEMA_PATH.write_text(json.dumps(feature_schema, indent=2), encoding="utf-8")
print(f"Schema guardado: {len(feature_schema)} features")
print(f" {feature_schema[:5]}")

 Modelo guardado: mlops_artifacts_v2/model_5tx.pkl (1,451,449 bytes)
FeatureBuilder guardado: mlops_artifacts_v2/feature_builder.pkl (14,978 bytes)
 Config guardada: mlops_artifacts_v2/cfg.pkl
Schema guardado: 64 features
 ['userTypeIdentifier', 'accountLevel', 'IsActive', 'phn_confir', 'email_confir']


In [31]:
import os
from pathlib import Path

ARTIFACTS_DIR = Path("mlops_artifacts_v2")

for f in ARTIFACTS_DIR.glob("*"):
    size = os.path.getsize(f)
    print(f"{f.name}: {size:,} bytes")

predicciones_5tx_30d_v27.csv: 156,484,759 bytes
df_raw_column_catalog_OLD_MODEL.csv: 1,876 bytes
model_5tx_old.pkl: 1,445,064 bytes
predicciones_5tx_30d_v22.csv: 153,724,527 bytes
predicciones_5tx_30d_v15.csv: 143,322,786 bytes
feature_schema.json: 1,088 bytes
predicciones_5tx_30d_v11.csv: 141,168,771 bytes
predicciones_5tx_30d_v13.csv: 142,617,294 bytes
predicciones_5tx_30d_v26.csv: 156,057,967 bytes
predicciones_5tx_30d_v12.csv: 142,331,544 bytes
predicciones_5tx_30d_v14.csv: 142,960,433 bytes
predicciones_5tx_30d_v18.csv: 145,311,546 bytes
model_5tx.pkl: 1,451,449 bytes
predicciones_5tx_30d_v10.csv: 140,818,174 bytes
df_raw_suspect_columns_audit_OLD_MODEL.csv: 676 bytes
predicciones_5tx_30d_v25.csv: 155,686,810 bytes
predicciones_5tx_30d_v19.csv: 145,726,732 bytes
artifact_audit_used_for_predictions_OLD_MODEL.csv: 367 bytes
predicciones_5tx_30d_v24.csv: 154,480,738 bytes
predicciones_5tx_30d_v31.csv: 159,708,729 bytes
df_export_5tx_30d_2026-04-30_20:40:09.csv: 159,707,892 bytes
df_m

# MLOps inference

In [32]:
import json, sys, platform, cloudpickle  
from pathlib import Path
from datetime import datetime, timezone

In [33]:
ARTIFACTS_DIR = Path("mlops_artifacts_v2")
MODEL_PATH = ARTIFACTS_DIR / "model_5tx.pkl"
FB_PATH = ARTIFACTS_DIR / "feature_builder.pkl"
CFG_PATH = ARTIFACTS_DIR / "cfg.pkl"
SCHEMA_PATH = ARTIFACTS_DIR / "feature_schema.json"

In [34]:
# Cargar Feature Builder
with open(FB_PATH, "rb") as f:
    fb_loaded = cloudpickle.load(f)
print(f"FeatureBuilder: {type(fb_loaded).__name__}")

# Cargar Modelo
with open(MODEL_PATH, "rb") as f:
    model_loaded = cloudpickle.load(f)
print(f"Modelo: {type(model_loaded).__name__}")

# Cargar Config
cfg_loaded = None
if CFG_PATH.exists():
    with open(CFG_PATH, "rb") as f:
        cfg_loaded = cloudpickle.load(f)
    print(f"Config: {type(cfg_loaded).__name__}")
else:
    print("Config no encontrada (opcional)")

# Cargar Schema
schema_loaded = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
print(f"Schema: {len(schema_loaded)} features")

# Validación de schema no debe estar vacio
if len(schema_loaded) == 0:
    raise ValueError("CRÍTICO: feature_schema.json está vacío!")


FeatureBuilder: FeatureBuilder
Modelo: LGBMClassifier
Config: Config
Schema: 64 features


In [35]:
# Extraer features del modelo
if hasattr(model_loaded, "feature_name_") and model_loaded.feature_name_:
    model_features = list(model_loaded.feature_name_)
elif hasattr(model_loaded, "booster_"):
    model_features = list(model_loaded.booster_.feature_name())
else:
    model_features = []
    print("No se pudieron extraer features del modelo")

# Comparar
if model_features:
    schema_set = set(schema_loaded)
    model_set = set(model_features)
    
    missing_in_schema = model_set - schema_set
    extra_in_schema = schema_set - model_set
    
    if missing_in_schema:
        print(f"Features en modelo pero no en schema: {len(missing_in_schema)}")
    if extra_in_schema:
        print(f"Features en schema pero no en modelo: {len(extra_in_schema)}")
    
    if not missing_in_schema and not extra_in_schema:
        print(f"Schema consistente: {len(model_features)} features")


Features en modelo pero no en schema: 64
Features en schema pero no en modelo: 64


In [36]:
from google.cloud import bigquery
project_id = cfg_loaded.project_id if cfg_loaded else "spin-aip-singularity-comp-sb"
table_fqn = cfg_loaded.table_fqn if cfg_loaded else "spin-aip-singularity-comp-sb.model_activation.model_5trx_test_v31"

client = bigquery.Client(project=project_id)
query = f"SELECT * FROM `{table_fqn}`"

data_scoring = client.query(query).to_dataframe()
print(f"Datos cargados: {data_scoring.shape}")


Datos cargados: (3834967, 37)


In [37]:
# Filtrar solo activados (como en training)
if "label_activated_30d" in data_scoring.columns:
    n_before = len(data_scoring)
    data_scoring = data_scoring[data_scoring["label_activated_30d"] == 1].copy()
    print(f"Filtro activados: {n_before} -> {len(data_scoring)}")

Filtro activados: 3834967 -> 2371692


In [38]:
# Aplicar FeatureBuilder
bunch = fb_loaded.transform(data_scoring)
X = bunch.X

print(f"Features generadas: {X.shape}")

Features generadas: (2371692, 64)


In [39]:
# Alinear con schema del modelo (maneja features faltantes/extras)
X_aligned = X.reindex(columns=schema_loaded, fill_value=0)
print(f"Features alineadas: {X_aligned.shape}")

# Convertir a numpy
X_np = X_aligned.values.astype("float32")
print(f"Array numpy: {X_np.shape}, dtype: {X_np.dtype}")

Features alineadas: (2371692, 64)
Array numpy: (2371692, 64), dtype: float32


In [40]:
# Predicción
proba = model_loaded.predict_proba(X_np)[:, 1]

print(f"Predicciones: {len(proba)}")
print(f"   - Min:  {proba.min():.4f}")
print(f"   - Max:  {proba.max():.4f}")
print(f"   - Mean: {proba.mean():.4f}")
print(f"   - Std:  {proba.std():.4f}")

Predicciones: 2371692
   - Min:  0.0004
   - Max:  0.9999
   - Mean: 0.6208
   - Std:  0.3873


In [41]:
#  Output
import pandas as pd

output = pd.DataFrame({"p_5tx_30": proba})

# Agregar user_id 
if "user_id" in data_scoring.columns:
    output.insert(0, "user_id", data_scoring["user_id"].values)

print(f"Output shape: {output.shape}")
print(f"\n Preview:")
output.head(10)

Output shape: (2371692, 2)

 Preview:


,user_id,p_5tx_30
0,0005eaa7-4309-4a9d-a43d-7d01a7849220,0.9976
1,00073792-5c60-4ab1-a712-719f260fac54,0.9974
2,000bb768-1e8d-4440-b134-215ff1eaf51a,0.9992
3,000d8f79-8094-4489-a6d4-5c409199014e,0.9927
4,000f24ac-dafc-40ad-9032-7fcd70f77d4d,0.9305
5,0010d378-a80f-4371-a218-41c7e601f775,0.9888
6,0011738c-d3d2-4947-b034-95e3b70636b6,0.9961
7,00119770-0f4b-454c-b583-b6f32d34e33f,0.9968
8,0012df96-9c88-4ec8-863c-60ba0096b2a4,0.9995
9,0018fc64-eb72-4f21-87a3-510da3198c36,0.9896


In [42]:
# 1. Verificar varianza en las features
print("=" * 60)
print("DIAGNÓSTICO DE FEATURES")
print("=" * 60)

# Varianza por columna
print(f"\nShape de X_aligned: {X_aligned.shape}")
print(f"Shape de X (antes de align): {X.shape}")

# ¿Cuántas columnas tienen varianza > 0?
variances = X_aligned.var()
cols_with_variance = (variances > 0).sum()
cols_all_zero = (variances == 0).sum()

print(f"\nColumnas con varianza > 0: {cols_with_variance}")
print(f"Columnas con varianza = 0 (constantes): {cols_all_zero}")

# 2. ¿Cuántas columnas del FB coinciden con el schema?
fb_cols = set(X.columns)
schema_cols = set(schema_loaded)

matching = fb_cols & schema_cols
missing_in_fb = schema_cols - fb_cols
extra_in_fb = fb_cols - schema_cols

print(f"\n📊 Comparación FB vs Schema:")
print(f"   - Columnas que coinciden: {len(matching)}")
print(f"   - En schema pero NO en FB: {len(missing_in_fb)}")
print(f"   - En FB pero NO en schema: {len(extra_in_fb)}")

if len(missing_in_fb) > 0:
    print(f"\nColumnas faltantes (se llenaron con 0):")
    print(f"   {list(missing_in_fb)[:10]}...")  # Primeras 10

# 3. Preview de los datos
print(f"\nPreview de X_aligned (primeras 5 filas, 10 cols):")
print(X_aligned.iloc[:5, :10])

# 4. ¿Todo es cero?
total_zeros = (X_aligned == 0).sum().sum()
total_cells = X_aligned.shape[0] * X_aligned.shape[1]
pct_zeros = total_zeros / total_cells * 100
print(f"\n% de celdas que son cero: {pct_zeros:.2f}%")

DIAGNÓSTICO DE FEATURES

Shape de X_aligned: (2371692, 64)
Shape de X (antes de align): (2371692, 64)

Columnas con varianza > 0: 61
Columnas con varianza = 0 (constantes): 3

📊 Comparación FB vs Schema:
   - Columnas que coinciden: 64
   - En schema pero NO en FB: 0
   - En FB pero NO en schema: 0

Preview de X_aligned (primeras 5 filas, 10 cols):
   userTypeIdentifier  accountLevel  IsActive  phn_confir  email_confir  \
0              3.0000        2.0000         1      0.0000        0.0000   
2              3.0000        2.0000         1      0.0000        0.0000   
3              3.0000        3.0000         1      0.0000        0.0000   
4              3.0000        3.0000         1      0.0000        0.0000   
5              3.0000        2.0000         1      0.0000        0.0000   

   phn_confir_d7  email_confir_d7  both_confir_d7  has_premia  \
0         1.0000           0.0000          0.0000      1.0000   
2         1.0000           1.0000          1.0000      1.0000   
3  

In [43]:
# Ver qué tiene cada uno
print("Schema guardado (primeras 10):")
print(schema_loaded[:10])

print("\nColumnas del FeatureBuilder (primeras 10):")
print(list(X.columns)[:10])

Schema guardado (primeras 10):
['userTypeIdentifier', 'accountLevel', 'IsActive', 'phn_confir', 'email_confir', 'phn_confir_d7', 'email_confir_d7', 'both_confir_d7', 'has_premia', 'label_activated_30d']

Columnas del FeatureBuilder (primeras 10):
['userTypeIdentifier', 'accountLevel', 'IsActive', 'phn_confir', 'email_confir', 'phn_confir_d7', 'email_confir_d7', 'both_confir_d7', 'has_premia', 'label_activated_30d']


In [44]:
# ============================================================
# FINAL EXPORT - IDs + FEATURES UTILIZADAS + SCORE DEL MODELO
# Modelo 5TX 30D - LightGBM
# ============================================================

from pathlib import Path
import re
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. Validación de objetos esperados
# ------------------------------------------------------------

required_objects = [
    "data_scoring",
    "schema_loaded",
    "model_loaded"
]

missing_objects = [obj for obj in required_objects if obj not in globals()]
if missing_objects:
    raise NameError(
        f"Faltan objetos requeridos en memoria: {missing_objects}. "
        "Corre primero las celdas de carga de artefactos, carga de data y scoring."
    )

# Si X_aligned o proba no existen, se regeneran.
# Esto hace que la celda sea robusta si la corres limpia después de cargar artefactos.
if "X_aligned" not in globals():
    if "fb_loaded" not in globals():
        raise NameError(
            "No existe X_aligned ni fb_loaded. "
            "Necesitas cargar el FeatureBuilder o correr la celda donde se genera X_aligned."
        )
    
    print("X_aligned no existe. Regenerando features con fb_loaded.transform(data_scoring)...")
    bunch_scoring = fb_loaded.transform(data_scoring)
    X_scoring = bunch_scoring.X
    
    X_aligned = X_scoring.reindex(columns=schema_loaded, fill_value=0)
    print(f"Features regeneradas y alineadas: {X_aligned.shape}")

if "proba" not in globals():
    print("proba no existe. Regenerando scores con model_loaded.predict_proba...")
    X_np = X_aligned.values.astype("float32")
    proba = model_loaded.predict_proba(X_np)[:, 1]
    print(f"Scores regenerados: {len(proba):,}")

# ------------------------------------------------------------
# 1. Checks duros de consistencia
# ------------------------------------------------------------

if len(data_scoring) != len(X_aligned):
    raise ValueError(
        f"Inconsistencia de filas: data_scoring={len(data_scoring):,}, "
        f"X_aligned={len(X_aligned):,}. Revisa filtros o reset_index."
    )

if len(proba) != len(X_aligned):
    raise ValueError(
        f"Inconsistencia de scores: proba={len(proba):,}, "
        f"X_aligned={len(X_aligned):,}."
    )

if list(X_aligned.columns) != list(schema_loaded):
    raise ValueError(
        "El orden de columnas de X_aligned no coincide con schema_loaded. "
        "Esto puede romper el score porque LightGBM fue entrenado por posición."
    )

print("Checks OK:")
print(f" - Filas scoring       : {len(data_scoring):,}")
print(f" - Features del modelo : {len(schema_loaded):,}")
print(f" - Scores generados    : {len(proba):,}")

# ------------------------------------------------------------
# 2. Construcción de columnas ID / trazabilidad
# ------------------------------------------------------------

id_regex = re.compile(
    r"(^id$|_id$|identifier$|accountid$|userid$|user_id$|member_id$|spin_user_id$)",
    flags=re.IGNORECASE
)

id_like_cols = [
    c for c in data_scoring.columns
    if id_regex.search(str(c))
]

preferred_id_cols = [
    "user_id",
    "premia_accountid",
    "channelUserIdentifier",
    "userTypeIdentifier",
    "signup_date",
    "signup_ts"
]

id_cols = []
for c in preferred_id_cols + id_like_cols:
    if c in data_scoring.columns and c not in id_cols:
        id_cols.append(c)

if "user_id" not in id_cols:
    raise ValueError(
        "No encontré user_id en data_scoring. "
        "Sin user_id la tabla no es operable para CRM/MLOps."
    )

ids_df = data_scoring[id_cols].reset_index(drop=True).copy()

# ID técnico de corrida / fila para trazabilidad
tz_local = getattr(cfg_loaded, "tz_local", "America/Mexico_City") if "cfg_loaded" in globals() else "America/Mexico_City"
score_ts = pd.Timestamp.now(tz=tz_local)

ids_df.insert(0, "model_row_id", np.arange(len(ids_df), dtype=np.int64))
ids_df["score_generated_ts"] = score_ts

# ------------------------------------------------------------
# 3. Tabla de scores
# ------------------------------------------------------------

scores_df = pd.DataFrame({
    "p_5tx_30d": proba.astype("float32")
})

# Ranking operativo
scores_df["score_rank_desc"] = scores_df["p_5tx_30d"].rank(
    method="first",
    ascending=False
).astype("int64")

# Decil: 10 = mayor propensión, 1 = menor propensión
scores_df["score_decile"] = pd.qcut(
    scores_df["p_5tx_30d"].rank(method="first"),
    q=10,
    labels=False,
    duplicates="drop"
).astype("int16") + 1

# Invertimos para que decil 10 sea top score
scores_df["score_decile"] = 11 - scores_df["score_decile"]

# ------------------------------------------------------------
# 4. Features utilizadas por el modelo
# ------------------------------------------------------------

# Prefijo para evitar colisión entre IDs y features.
# Ejemplo: userTypeIdentifier puede aparecer como ID y como feature.
features_df = X_aligned.reset_index(drop=True).copy()
features_df.columns = [f"feature__{c}" for c in features_df.columns]

# ------------------------------------------------------------
# 5. Tabla final wide: IDs + score + todas las features usadas
# ------------------------------------------------------------

df_model_scoring_wide = pd.concat(
    [
        ids_df.reset_index(drop=True),
        scores_df.reset_index(drop=True),
        features_df.reset_index(drop=True)
    ],
    axis=1
)

print("\nTabla final generada:")
print(f" - Shape: {df_model_scoring_wide.shape}")
print(f" - IDs incluidos: {id_cols}")
print(f" - Features incluidas: {len(schema_loaded)}")
print(f" - Score column: p_5tx_30d")

display(df_model_scoring_wide.head())

# ------------------------------------------------------------
# 6. Catálogo de features usadas por el modelo
# ------------------------------------------------------------

df_feature_catalog = pd.DataFrame({
    "feature_id": [f"F{i+1:03d}" for i in range(len(schema_loaded))],
    "model_input_position": np.arange(len(schema_loaded), dtype=np.int16),
    "feature_name": list(schema_loaded),
    "wide_table_column": [f"feature__{c}" for c in schema_loaded]
})

# Importancias del modelo por posición.
# Nota: el modelo fue entrenado con numpy, por eso LightGBM puede mostrar Column_0, Column_1...
# Aquí hacemos el mapeo correcto por posición contra schema_loaded.
if hasattr(model_loaded, "feature_importances_"):
    split_importance = np.asarray(model_loaded.feature_importances_)
    if len(split_importance) == len(schema_loaded):
        df_feature_catalog["importance_split"] = split_importance
    else:
        df_feature_catalog["importance_split"] = np.nan
else:
    df_feature_catalog["importance_split"] = np.nan

if hasattr(model_loaded, "booster_"):
    try:
        gain_importance = model_loaded.booster_.feature_importance(importance_type="gain")
        if len(gain_importance) == len(schema_loaded):
            df_feature_catalog["importance_gain"] = gain_importance
        else:
            df_feature_catalog["importance_gain"] = np.nan
    except Exception:
        df_feature_catalog["importance_gain"] = np.nan
else:
    df_feature_catalog["importance_gain"] = np.nan

# Ranking de importancia
if "importance_gain" in df_feature_catalog.columns and df_feature_catalog["importance_gain"].notna().any():
    df_feature_catalog["importance_rank"] = (
        df_feature_catalog["importance_gain"]
        .rank(method="dense", ascending=False)
        .astype("Int64")
    )
else:
    df_feature_catalog["importance_rank"] = (
        df_feature_catalog["importance_split"]
        .rank(method="dense", ascending=False)
        .astype("Int64")
    )

df_feature_catalog = df_feature_catalog.sort_values(
    ["importance_rank", "model_input_position"],
    ascending=[True, True]
).reset_index(drop=True)

print("\nCatálogo de features generado:")
display(df_feature_catalog.head(30))

# ------------------------------------------------------------
# 7. Alerta de leakage / columnas sospechosas
# ------------------------------------------------------------

leakage_keywords = [
    "label",
    "target",
    "tx_30d",
    "activation",
    "first_tx",
    "latest_tx",
    "days_to_first",
    "from_activation",
    "p_tx_"
]

suspect_features = [
    f for f in schema_loaded
    if any(k.lower() in f.lower() for k in leakage_keywords)
]

df_suspect_features = df_feature_catalog[
    df_feature_catalog["feature_name"].isin(suspect_features)
].copy()

if len(df_suspect_features) > 0:
    print("\n⚠️ ALERTA: El modelo actual está usando features potencialmente leaky o post-window.")
    print("Esto sirve para auditar el modelo actual, pero NO debería pasar a producción sin revisión.")
    display(df_suspect_features)
else:
    print("\nOK: No se detectaron features sospechosas por keywords de leakage.")

# ------------------------------------------------------------
# 8. Guardar artefactos
# ------------------------------------------------------------

OUTPUT_DIR = Path("mlops_artifacts_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

wide_parquet_path = OUTPUT_DIR / "df_model_scoring_wide_ids_features_scores.parquet"
wide_csv_path = OUTPUT_DIR / "df_model_scoring_wide_ids_features_scores.csv"
feature_catalog_path = OUTPUT_DIR / "df_feature_catalog_model_5tx.csv"
suspect_features_path = OUTPUT_DIR / "df_suspect_features_leakage_audit.csv"

df_model_scoring_wide.to_parquet(wide_parquet_path, index=False)
df_model_scoring_wide.to_csv(wide_csv_path, index=False)
df_feature_catalog.to_csv(feature_catalog_path, index=False)
df_suspect_features.to_csv(suspect_features_path, index=False)

print("\nArchivos guardados:")
print(f" - Wide parquet      : {wide_parquet_path}")
print(f" - Wide csv          : {wide_csv_path}")
print(f" - Feature catalog   : {feature_catalog_path}")
print(f" - Leakage audit     : {suspect_features_path}")

# ------------------------------------------------------------
# 9. Resumen ejecutivo de score
# ------------------------------------------------------------

score_summary = df_model_scoring_wide["p_5tx_30d"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame("p_5tx_30d")

display(score_summary)

print("\nDistribución por decil:")
display(
    df_model_scoring_wide
    .groupby("score_decile", as_index=False)
    .agg(
        n_users=("user_id", "count"),
        min_score=("p_5tx_30d", "min"),
        mean_score=("p_5tx_30d", "mean"),
        max_score=("p_5tx_30d", "max")
    )
    .sort_values("score_decile", ascending=False)
)

Checks OK:
 - Filas scoring       : 2,371,692
 - Features del modelo : 64
 - Scores generados    : 2,371,692

Tabla final generada:
 - Shape: (2371692, 75)
 - IDs incluidos: ['user_id', 'premia_accountid', 'channelUserIdentifier', 'userTypeIdentifier', 'signup_date', 'signup_ts']
 - Features incluidas: 64
 - Score column: p_5tx_30d


,model_row_id,user_id,premia_accountid,channelUserIdentifier,userTypeIdentifier,signup_date,signup_ts,score_generated_ts,p_5tx_30d,score_rank_desc,score_decile,feature__userTypeIdentifier,feature__accountLevel,feature__IsActive,feature__phn_confir,feature__email_confir,feature__phn_confir_d7,feature__email_confir_d7,feature__both_confir_d7,feature__has_premia,feature__label_activated_30d,feature__tx_30d_amount,feature__first_tx_amount,feature__lifespan_days,feature__days_since_last,feature__days_to_first_activation,feature__gender_bin,feature__user_type_tri,feature__channel_detail_code,feature__birth_bucket,feature__age_years,feature__signup_dow,feature__signup_week,feature__signup_month,feature__signup_daypart,feature__is_holiday_mx,feature__near_payday_any,feature__near_payday_1st,feature__near_payday_15,feature__near_payday_eom,feature__state_AG,feature__state_BC,feature__state_BS,feature__state_CH,feature__state_CL,feature__state_CM,feature__state_CO,feature__state_CS,feature__state_DF,feature__state_DG,feature__state_EM,feature__state_GR,feature__state_GT,feature__state_HG,feature__state_JA,feature__state_MI,feature__state_MO,feature__state_NA,feature__state_NL,feature__state_OA,feature__state_OT,feature__state_PU,feature__state_QR,feature__state_QT,feature__state_SI,feature__state_SL,feature__state_SO,feature__state_TB,feature__state_TL,feature__state_TM,feature__state_VE,feature__state_YU,feature__state_ZA,feature__card_linked_before_signup,feature__card_linked_lag_days
0,0,0005eaa7-4309-4a9d-a43d-7d01a7849220,2WHVMJB,1,3,2025-12-07,2025-12-07 20:16:48.643000+00:00,2026-04-30 14:41:21.986990-06:00,0.9976,171997,1,3.0000,2.0000,1,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,1.0000,5.0462,-0.3963,-0.0138,-0.1319,-0.2500,0.0000,0.0000,2.0000,1.0000,0.5308,6.0000,1.0000,1.1667,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
1,1,00073792-5c60-4ab1-a712-719f260fac54,7F5W5LZ,1,3,2025-09-26,2025-09-26 19:47:33.124000+00:00,2026-04-30 14:41:21.986990-06:00,0.9974,188017,1,3.0000,2.0000,1,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,2.1655,1.6870,0.2627,-0.0110,0.0000,1.0000,0.0000,2.0000,5.0000,0.1097,4.0000,0.6429,0.6667,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000
2,2,000bb768-1e8d-4440-b134-215ff1eaf51a,JPTOCIW,1,3,2025-11-25,2025-11-26 02:30:29.093000+00:00,2026-04-30 14:41:21.986990-06:00,0.9992,45640,1,3.0000,3.0000,1,0.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,13.8297,0.5500,0.0092,-0.0769,0.2500,0.0000,0.0000,2.0000,5.0000,-0.3281,1.0000,0.9643,1.0000,2.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000
3,3,000d8f79-8094-4489-a6d4-5c409199014e,UCHGI6G,1,3,2025-02-05,2025-02-06 03:13:05.792000+00:00,2026-04-30 14:41:21.986990-06:00,0.9927,362313,2,3.0000,3.0000,1,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,1.0000,1.2536,-0.3969,1.3871,-0.1209,-0.2500,1.0000,0.0000,2.0000,1.0000,-0.4494,2.0000,-0.5357,-0.5000,2.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000
4,4,000f24ac-dafc-40ad-9032-7fcd70f77d4d,CDHMAUK,1,3,2025-12-04,2025-12-04 22:42:55.637000+00:00,2026-04-30 14:41:21.986990-06:00


Catálogo de features generado:


,feature_id,model_input_position,feature_name,wide_table_column,importance_split,importance_gain,importance_rank
0,F011,10,tx_30d_amount,feature__tx_30d_amount,4066,9567648.6844,1
1,F012,11,first_tx_amount,feature__first_tx_amount,3968,3855099.4236,2
2,F013,12,lifespan_days,feature__lifespan_days,710,1634355.8690,3
3,F020,19,age_years,feature__age_years,513,288062.7398,4
4,F015,14,days_to_first_activation,feature__days_to_first_activation,484,227826.9481,5
5,F014,13,days_since_last,feature__days_since_last,248,139735.0316,6
6,F006,5,phn_confir_d7,feature__phn_confir_d7,129,131134.8481,7
7,F001,0,userTypeIdentifier,feature__userTypeIdentifier,219,122304.2168,8
8,F019,18,birth_bucket,feature__birth_bucket,176,54263.1152,9
9,F007,6,email_confir_d7,feature__email_confir_d7,47,38217.2427,10



⚠️ ALERTA: El modelo actual está usando features potencialmente leaky o post-window.
Esto sirve para auditar el modelo actual, pero NO debería pasar a producción sin revisión.


,feature_id,model_input_position,feature_name,wide_table_column,importance_split,importance_gain,importance_rank
0,F011,10,tx_30d_amount,feature__tx_30d_amount,4066,9567648.6844,1
1,F012,11,first_tx_amount,feature__first_tx_amount,3968,3855099.4236,2
4,F015,14,days_to_first_activation,feature__days_to_first_activation,484,227826.9481,5
60,F010,9,label_activated_30d,feature__label_activated_30d,0,0.0000,59



Archivos guardados:
 - Wide parquet      : mlops_artifacts_v2/df_model_scoring_wide_ids_features_scores.parquet
 - Wide csv          : mlops_artifacts_v2/df_model_scoring_wide_ids_features_scores.csv
 - Feature catalog   : mlops_artifacts_v2/df_feature_catalog_model_5tx.csv
 - Leakage audit     : mlops_artifacts_v2/df_suspect_features_leakage_audit.csv


,p_5tx_30d
count,2371692.0000
mean,0.6208
std,0.3873
min,0.0004
1%,0.0013
5%,0.0074
10%,0.0196
25%,0.1914
50%,0.8065
75%,0.9783



Distribución por decil:


,score_decile,n_users,min_score,mean_score,max_score
9,10,237170,0.0004,0.0081,0.0196
8,9,237169,0.0196,0.0524,0.1032
7,8,237169,0.1032,0.1976,0.3162
6,7,237169,0.3162,0.4597,0.6009
5,6,237169,0.6009,0.7130,0.8065
4,5,237169,0.8065,0.8664,0.9134
3,4,237169,0.9134,0.9421,0.9644
2,3,237169,0.9644,0.9774,0.9873
1,2,237169,0.9873,0.9926,0.9964
0,1,237170,0.9964,0.9983,0.9999


In [45]:
# ============================================================
# FINAL EXPORT - TABLA ORIGINAL + SCORE DEL MODELO
# FORZANDO USO DE: mlops_artifacts_v2/model_5tx_old.pkl
# Modelo 5TX 30D - LightGBM
# ============================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import cloudpickle
import joblib

# ------------------------------------------------------------
# 0. Configuración de artefactos
# ------------------------------------------------------------

ARTIFACTS_DIR = Path("mlops_artifacts_v2")

OLD_MODEL_PATH = ARTIFACTS_DIR / "model_5tx_old.pkl"

if not OLD_MODEL_PATH.exists():
    raise FileNotFoundError(f"No existe el modelo OLD en: {OLD_MODEL_PATH}")

# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def file_sha256(path: Path) -> str:
    """
    Calcula hash SHA256 del artifact para trazabilidad/auditoría.
    """
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def load_pickle_model(path: Path):
    """
    Carga modelo con cloudpickle y fallback a joblib.
    """
    try:
        with open(path, "rb") as f:
            return cloudpickle.load(f)
    except Exception:
        return joblib.load(path)

# ------------------------------------------------------------
# 2. Validación de objetos requeridos
# ------------------------------------------------------------

required_objects = [
    "data_scoring",
    "schema_loaded",
    "fb_loaded"
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"Faltan objetos requeridos en memoria: {missing_objects}. "
        "Corre primero las celdas de carga de datos, FeatureBuilder y schema."
    )

# ------------------------------------------------------------
# 3. Cargar EXPLÍCITAMENTE model_5tx_old.pkl como modelo activo
# ------------------------------------------------------------

model_loaded = load_pickle_model(OLD_MODEL_PATH)

model_artifact_name = "model_5tx_old.pkl"
model_artifact_path = str(OLD_MODEL_PATH)
model_artifact_sha256 = file_sha256(OLD_MODEL_PATH)

print("Modelo activo cargado:")
print(f" - artifact_name : {model_artifact_name}")
print(f" - artifact_path : {model_artifact_path}")
print(f" - sha256        : {model_artifact_sha256}")
print(f" - object_type   : {type(model_loaded)}")

# ------------------------------------------------------------
# 4. Regenerar SIEMPRE X_aligned
#    Esto evita reutilizar una matriz vieja en memoria.
# ------------------------------------------------------------

print("\nRegenerando features transformadas desde tabla original...")

bunch = fb_loaded.transform(data_scoring)
X = bunch.X

X_aligned = X.reindex(columns=schema_loaded, fill_value=0)

if list(X_aligned.columns) != list(schema_loaded):
    raise ValueError(
        "El orden de columnas de X_aligned no coincide con schema_loaded. "
        "Esto puede romper la inferencia porque LightGBM depende del orden de columnas."
    )

print("Features transformadas y alineadas:")
print(f" - X original    : {X.shape}")
print(f" - X_aligned     : {X_aligned.shape}")
print(f" - schema_loaded : {len(schema_loaded)} features")

# ------------------------------------------------------------
# 5. Forzar regeneración de proba con model_5tx_old.pkl
#    NO usar: if 'proba' not in globals()
# ------------------------------------------------------------

print("\nRegenerando scores usando model_5tx_old.pkl...")

X_np = X_aligned.values.astype("float32")

proba = model_loaded.predict_proba(X_np)[:, 1]

print("Scores generados:")
print(f" - n_scores   : {len(proba):,}")
print(f" - min_score  : {proba.min():.8f}")
print(f" - mean_score : {proba.mean():.8f}")
print(f" - max_score  : {proba.max():.8f}")

# ------------------------------------------------------------
# 6. Checks duros de consistencia
# ------------------------------------------------------------

if len(data_scoring) != len(proba):
    raise ValueError(
        f"Inconsistencia de filas: data_scoring={len(data_scoring):,}, "
        f"proba={len(proba):,}. Revisa filtros antes/después del scoring."
    )

if len(data_scoring) != len(X_aligned):
    raise ValueError(
        f"Inconsistencia de filas: data_scoring={len(data_scoring):,}, "
        f"X_aligned={len(X_aligned):,}."
    )

if hasattr(model_loaded, "n_features_in_"):
    if model_loaded.n_features_in_ != X_aligned.shape[1]:
        raise ValueError(
            f"Inconsistencia de features: modelo espera {model_loaded.n_features_in_}, "
            f"pero X_aligned tiene {X_aligned.shape[1]}."
        )

print("\nChecks OK:")
print(f" - Filas originales scoreadas : {len(data_scoring):,}")
print(f" - Features modelo internas   : {X_aligned.shape[1]:,}")
print(f" - Scores generados           : {len(proba):,}")
print(f" - Modelo usado               : {model_artifact_name}")

# ------------------------------------------------------------
# 7. Crear tabla final con FEATURES CRUDAS / TABLA ORIGINAL
# ------------------------------------------------------------

df_model_scoring_raw = data_scoring.reset_index(drop=True).copy()

# Evitar duplicados si la tabla ya trae score de una corrida anterior
cols_to_drop_if_exist = [
    "model_row_id",
    "score_generated_ts",
    "p_5tx_30d",
    "score_rank_desc",
    "score_decile",
    "model_artifact_name",
    "model_artifact_path",
    "model_artifact_sha256"
]

df_model_scoring_raw = df_model_scoring_raw.drop(
    columns=[c for c in cols_to_drop_if_exist if c in df_model_scoring_raw.columns],
    errors="ignore"
)

# Timestamp de scoring
tz_local = (
    getattr(cfg_loaded, "tz_local", "America/Mexico_City")
    if "cfg_loaded" in globals() and cfg_loaded is not None
    else "America/Mexico_City"
)

score_ts = pd.Timestamp.now(tz=tz_local)

# Metadata operativa
df_model_scoring_raw.insert(
    0,
    "model_row_id",
    np.arange(len(df_model_scoring_raw), dtype=np.int64)
)

df_model_scoring_raw.insert(
    1,
    "score_generated_ts",
    score_ts
)

# Score
df_model_scoring_raw["p_5tx_30d"] = proba.astype("float32")

# Trazabilidad del artifact usado
df_model_scoring_raw["model_artifact_name"] = model_artifact_name
df_model_scoring_raw["model_artifact_path"] = model_artifact_path
df_model_scoring_raw["model_artifact_sha256"] = model_artifact_sha256

# Ranking operativo: 1 = mayor probabilidad
df_model_scoring_raw["score_rank_desc"] = (
    df_model_scoring_raw["p_5tx_30d"]
    .rank(method="first", ascending=False)
    .astype("int64")
)

# Decil operativo: 10 = mayor propensión, 1 = menor propensión
df_model_scoring_raw["score_decile"] = pd.qcut(
    df_model_scoring_raw["p_5tx_30d"].rank(method="first"),
    q=10,
    labels=False,
    duplicates="drop"
).astype("int16") + 1

df_model_scoring_raw["score_decile"] = 11 - df_model_scoring_raw["score_decile"]

print("\nTabla final generada con columnas CRUDAS:")
print(f" - Shape: {df_model_scoring_raw.shape}")
print(f" - Columnas originales conservadas: {data_scoring.shape[1]}")
print(
    " - Columnas agregadas: "
    "model_row_id, score_generated_ts, p_5tx_30d, "
    "model_artifact_name, model_artifact_path, model_artifact_sha256, "
    "score_rank_desc, score_decile"
)

display(df_model_scoring_raw.head())

# ------------------------------------------------------------
# 8. Catálogo de columnas crudas exportadas
# ------------------------------------------------------------

model_output_cols = [
    "model_row_id",
    "score_generated_ts",
    "p_5tx_30d",
    "model_artifact_name",
    "model_artifact_path",
    "model_artifact_sha256",
    "score_rank_desc",
    "score_decile"
]

df_raw_column_catalog = pd.DataFrame({
    "column_position": np.arange(df_model_scoring_raw.shape[1]),
    "column_name": df_model_scoring_raw.columns,
    "dtype": [str(df_model_scoring_raw[c].dtype) for c in df_model_scoring_raw.columns],
    "source": np.where(
        df_model_scoring_raw.columns.isin(model_output_cols),
        "model_output",
        "original_table"
    )
})

display(df_raw_column_catalog)

# ------------------------------------------------------------
# 9. Auditoría rápida de columnas sensibles/leaky presentes
#    No las elimina: solo las marca porque son parte de la tabla original.
# ------------------------------------------------------------

leakage_keywords = [
    "label",
    "target",
    "tx_30d",
    "activation",
    "first_tx",
    "latest_tx",
    "days_to_first",
    "from_activation"
]

raw_suspect_cols = [
    c for c in data_scoring.columns
    if any(k.lower() in str(c).lower() for k in leakage_keywords)
]

df_raw_suspect_cols = pd.DataFrame({
    "raw_column": raw_suspect_cols,
    "reason": "possible_label_or_post_window_signal"
})

if len(df_raw_suspect_cols) > 0:
    print("\n⚠️ Columnas de la tabla original que parecen labels o señales post-window.")
    print("No se eliminan porque pediste tabla original, pero NO deberían usarse como features productivas.")
    display(df_raw_suspect_cols)
else:
    print("\nOK: No se detectaron columnas sospechosas por keywords en la tabla original.")

# ------------------------------------------------------------
# 10. Guardar artefactos
# ------------------------------------------------------------

OUTPUT_DIR = Path("mlops_artifacts_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_parquet_path = OUTPUT_DIR / "df_model_scoring_raw_original_table_plus_score_OLD_MODEL.parquet"
raw_csv_path = OUTPUT_DIR / "df_model_scoring_raw_original_table_plus_score_OLD_MODEL.csv"
raw_catalog_path = OUTPUT_DIR / "df_raw_column_catalog_OLD_MODEL.csv"
raw_suspect_path = OUTPUT_DIR / "df_raw_suspect_columns_audit_OLD_MODEL.csv"
artifact_audit_path = OUTPUT_DIR / "artifact_audit_used_for_predictions_OLD_MODEL.csv"

df_model_scoring_raw.to_parquet(raw_parquet_path, index=False)
df_model_scoring_raw.to_csv(raw_csv_path, index=False)
df_raw_column_catalog.to_csv(raw_catalog_path, index=False)
df_raw_suspect_cols.to_csv(raw_suspect_path, index=False)

df_artifact_audit = pd.DataFrame([{
    "model_artifact_name": model_artifact_name,
    "model_artifact_path": model_artifact_path,
    "model_artifact_sha256": model_artifact_sha256,
    "n_rows_scored": len(df_model_scoring_raw),
    "n_model_features": X_aligned.shape[1],
    "score_min": float(np.min(proba)),
    "score_mean": float(np.mean(proba)),
    "score_max": float(np.max(proba)),
    "score_generated_ts": score_ts
}])

df_artifact_audit.to_csv(artifact_audit_path, index=False)

print("\nArchivos guardados:")
print(f" - Tabla raw + score parquet : {raw_parquet_path}")
print(f" - Tabla raw + score CSV     : {raw_csv_path}")
print(f" - Catálogo columnas         : {raw_catalog_path}")
print(f" - Auditoría leakage         : {raw_suspect_path}")
print(f" - Auditoría artifact        : {artifact_audit_path}")

display(df_artifact_audit)

# ------------------------------------------------------------
# 11. Resumen ejecutivo del score
# ------------------------------------------------------------

score_summary = df_model_scoring_raw["p_5tx_30d"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame("p_5tx_30d")

display(score_summary)

print("\nDistribución por decil:")
display(
    df_model_scoring_raw
    .groupby("score_decile", as_index=False)
    .agg(
        n_users=("p_5tx_30d", "size"),
        min_score=("p_5tx_30d", "min"),
        mean_score=("p_5tx_30d", "mean"),
        max_score=("p_5tx_30d", "max")
    )
    .sort_values("score_decile", ascending=False)
)

Modelo activo cargado:
 - artifact_name : model_5tx_old.pkl
 - artifact_path : mlops_artifacts_v2/model_5tx_old.pkl
 - sha256        : c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8cc1a47573c45bf64f9
 - object_type   : <class 'lightgbm.sklearn.LGBMClassifier'>

Regenerando features transformadas desde tabla original...
Features transformadas y alineadas:
 - X original    : (2371692, 64)
 - X_aligned     : (2371692, 64)
 - schema_loaded : 64 features

Regenerando scores usando model_5tx_old.pkl...
Scores generados:
 - n_scores   : 2,371,692
 - min_score  : 0.00288018
 - mean_score : 0.66416867
 - max_score  : 0.99982221

Checks OK:
 - Filas originales scoreadas : 2,371,692
 - Features modelo internas   : 64
 - Scores generados           : 2,371,692
 - Modelo usado               : model_5tx_old.pkl

Tabla final generada con columnas CRUDAS:
 - Shape: (2371692, 45)
 - Columnas originales conservadas: 37
 - Columnas agregadas: model_row_id, score_generated_ts, p_5tx_30d, model_artifact_nam

,model_row_id,score_generated_ts,user_id,signup_date,signup_ts,userTypeIdentifier,channelUserIdentifier,accountLevel,stateName,gender,user_type,channelDetail,birth_date,birthState,Card_linked_date,IsActive,phn_confir,email_confir,phone_conf_ts,email_conf_ts,phn_confir_d7,email_confir_d7,both_confir_d7,premia_accountid,has_premia,activation_date_ever,activation_date_30d,label_activated_30d,tx_30d_count,tx_30d_amount,label_5tx_30d,first_tx_type,first_tx_amount,activation_channel,latest_tx_date,lifespan_days,days_since_last,tx_30d_from_activation,days_to_first_activation,p_5tx_30d,model_artifact_name,model_artifact_path,model_artifact_sha256,score_rank_desc,score_decile
0,0,2026-04-30 14:44:30.788190-06:00,0005eaa7-4309-4a9d-a43d-7d01a7849220,2025-12-07,2025-12-07 20:16:48.643000+00:00,3,1,2,NL,male,HYBRID,POS,1980-08-17,NL,2025-12-07,True,1,0,2025-12-08 02:32:18.850000+00:00,NaT,1,0,0,2WHVMJB,1,2025-12-07,2025-12-07,1,55,61742.0000,1,SPEI_CASH_IN,2.0000,SPEI/Transfer,2026-04-09,124,1,55,0,0.9977,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,150615,1
1,1,2026-04-30 14:44:30.788190-06:00,00073792-5c60-4ab1-a712-719f260fac54,2025-09-26,2025-09-26 19:47:33.124000+00:00,3,1,2,NL,female,HYBRID,POS,1989-08-11,VE,2025-09-26,True,1,1,2025-09-27 03:06:57.839000+00:00,2025-09-27 03:03:41.652000+00:00,1,1,1,7F5W5LZ,1,2025-09-27,2025-09-27,1,58,29216.3500,1,SPEI_CASH_IN,3960.3000,SPEI/Transfer,2026-03-29,184,12,61,1,0.9977,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,150998,1
2,2,2026-04-30 14:44:30.788190-06:00,000bb768-1e8d-4440-b134-215ff1eaf51a,2025-11-25,2025-11-26 02:30:29.093000+00:00,3,1,3,PU,male,HYBRID,POS,1999-04-29,OA,2025-11-25,True,1,1,2025-11-26 08:41:35.788000+00:00,2025-11-26 08:40:45.193000+00:00,1,1,1,JPTOCIW,1,2025-11-27,2025-11-27,1,42,160914.0000,1,SPEI_CASH_IN,1800.0000,SPEI/Transfer,2026-04-04,129,6,44,2,0.9992,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,25149,1
3,3,2026-04-30 14:44:30.788190-06:00,000d8f79-8094-4489-a6d4-5c409199014e,2025-02-05,2025-02-06 03:13:05.792000+00:00,3,1,3,NL,female,HYBRID,POS,2001-03-02,NL,2025-02-05,True,1,1,2025-02-06 09:36:36.453000+00:00,2025-03-06 08:55:57.868000+00:00,1,0,0,UCHGI6G,1,2025-02-05,2025-02-05,1,52,18920.7600,1,SPEI_CASH_IN,0.9400,SPEI/Transfer,2026-04-08,428,2,52,0,0.9942,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,322148,2
4,4,2026-04-30 14:44:30.788190-06:00,000f24ac-dafc-40ad-9032-7fcd70f77d4d,2025-12-04,2025-12-04 22:42:55.637000+00:00,3,1,2,SL,male,HYBRID,POS,1978-06-15,SL,2025-12-04,True,1,1,2025-12-05 05:02:51.935000+00:00,2026-01-09 13:12:17.566000+00:00,1,0,0,CDHMAUK,1,2025-12-10,2025-12-10,1,33,5975.2900,1,CASH_IN_AT_OXXO_QR,200.0000,QR,2026-04-08,120,2,42,6,0.9465,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,863713,4


,column_position,column_name,dtype,source
0,0,model_row_id,int64,model_output
1,1,score_generated_ts,"datetime64[us, America/Mexico_City]",model_output
2,2,user_id,object,original_table
3,3,signup_date,object,original_table
4,4,signup_ts,"datetime64[us, UTC]",original_table
5,5,userTypeIdentifier,Int64,original_table
6,6,channelUserIdentifier,Int64,original_table
7,7,accountLevel,Int64,original_table
8,8,stateName,object,original_table
9,9,gender,object,original_table



⚠️ Columnas de la tabla original que parecen labels o señales post-window.
No se eliminan porque pediste tabla original, pero NO deberían usarse como features productivas.


,raw_column,reason
0,activation_date_ever,possible_label_or_post_window_signal
1,activation_date_30d,possible_label_or_post_window_signal
2,label_activated_30d,possible_label_or_post_window_signal
3,tx_30d_count,possible_label_or_post_window_signal
4,tx_30d_amount,possible_label_or_post_window_signal
5,label_5tx_30d,possible_label_or_post_window_signal
6,first_tx_type,possible_label_or_post_window_signal
7,first_tx_amount,possible_label_or_post_window_signal
8,activation_channel,possible_label_or_post_window_signal
9,latest_tx_date,possible_label_or_post_window_signal



Archivos guardados:
 - Tabla raw + score parquet : mlops_artifacts_v2/df_model_scoring_raw_original_table_plus_score_OLD_MODEL.parquet
 - Tabla raw + score CSV     : mlops_artifacts_v2/df_model_scoring_raw_original_table_plus_score_OLD_MODEL.csv
 - Catálogo columnas         : mlops_artifacts_v2/df_raw_column_catalog_OLD_MODEL.csv
 - Auditoría leakage         : mlops_artifacts_v2/df_raw_suspect_columns_audit_OLD_MODEL.csv
 - Auditoría artifact        : mlops_artifacts_v2/artifact_audit_used_for_predictions_OLD_MODEL.csv


,model_artifact_name,model_artifact_path,model_artifact_sha256,n_rows_scored,n_model_features,score_min,score_mean,score_max,score_generated_ts
0,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,2371692,64,0.0029,0.6642,0.9998,2026-04-30 14:44:30.788190-06:00


,p_5tx_30d
count,2371692.0000
mean,0.6642
std,0.3575
min,0.0029
1%,0.0123
5%,0.0348
10%,0.0799
25%,0.3136
50%,0.8468
75%,0.9801



Distribución por decil:


,score_decile,n_users,min_score,mean_score,max_score
9,10,237170,0.0029,0.0380,0.0799
8,9,237169,0.0799,0.1399,0.2130
7,8,237169,0.2130,0.3167,0.4328
6,7,237169,0.4328,0.5597,0.6815
5,6,237169,0.6815,0.7730,0.8468
4,5,237169,0.8468,0.8928,0.9290
3,4,237169,0.9290,0.9511,0.9686
2,3,237169,0.9686,0.9795,0.9880
1,2,237169,0.9880,0.9928,0.9963
0,1,237170,0.9963,0.9980,0.9998


In [47]:
# ============================================================
# CARGA DIRECTA A BIGQUERY - MODO ROBUSTO
# FORZANDO USO DE: mlops_artifacts_v2/model_5tx_old.pkl
# Convierte casi todo a STRING para evitar errores de schema
# ============================================================

from pathlib import Path
import re
import hashlib
import cloudpickle
import joblib
import numpy as np
import pandas as pd
from google.cloud import bigquery


# 0. Configuración de destino BigQuery
# ------------------------------------------------------------

PROJECT_ID = "spin-aip-singularity-data-sb"
DATASET_ID = "Test_predictions_MLOps_30D_model"
TABLE_NAME = "tbl_model_5trx_scoring_mlop_test_v1"

TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"

# ------------------------------------------------------------
# 1. Configuración de artefactos
# ------------------------------------------------------------

ARTIFACTS_DIR = Path("mlops_artifacts_v2")
OLD_MODEL_PATH = ARTIFACTS_DIR / "model_5tx_old.pkl"

if not OLD_MODEL_PATH.exists():
    raise FileNotFoundError(f"No existe el modelo OLD en: {OLD_MODEL_PATH}")

# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def file_sha256(path: Path) -> str:
    """
    Calcula hash SHA256 del artifact para trazabilidad/auditoría.
    """
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def load_pickle_model(path: Path):
    """
    Carga modelo con cloudpickle y fallback a joblib.
    """
    try:
        with open(path, "rb") as f:
            return cloudpickle.load(f)
    except Exception:
        return joblib.load(path)


def clean_bq_colname(col):
    """
    Limpia nombres de columnas para BigQuery:
    - Solo letras, números y underscore
    - No empieza con número
    - Lowercase
    """
    col = str(col).strip()
    col = re.sub(r"[^A-Za-z0-9_]", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")

    if col == "":
        col = "unnamed_col"

    if col[0].isdigit():
        col = f"col_{col}"

    return col.lower()


def make_unique_columns(cols):
    """
    Evita columnas duplicadas después de limpiar nombres.
    """
    seen = {}
    final_cols = []

    for col in cols:
        if col not in seen:
            seen[col] = 0
            final_cols.append(col)
        else:
            seen[col] += 1
            final_cols.append(f"{col}_{seen[col]}")

    return final_cols

# ------------------------------------------------------------
# 3. Validación de objetos requeridos en memoria
# ------------------------------------------------------------

required_objects = [
    "data_scoring",
    "schema_loaded",
    "fb_loaded"
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(
        f"Faltan objetos requeridos en memoria: {missing_objects}. "
        "Corre primero las celdas de carga de datos, FeatureBuilder y schema."
    )

# ------------------------------------------------------------
# 4. Cargar EXPLÍCITAMENTE model_5tx_old.pkl como modelo activo
# ------------------------------------------------------------

model_loaded = load_pickle_model(OLD_MODEL_PATH)

model_artifact_name = "model_5tx_old.pkl"
model_artifact_path = str(OLD_MODEL_PATH)
model_artifact_sha256 = file_sha256(OLD_MODEL_PATH)

print("Modelo activo cargado:")
print(f" - artifact_name : {model_artifact_name}")
print(f" - artifact_path : {model_artifact_path}")
print(f" - sha256        : {model_artifact_sha256}")
print(f" - object_type   : {type(model_loaded)}")

# ------------------------------------------------------------
# 5. Regenerar SIEMPRE X_aligned
#    No reutilizar nada viejo en memoria.
# ------------------------------------------------------------

print("\nRegenerando features transformadas desde data_scoring...")

bunch = fb_loaded.transform(data_scoring)
X = bunch.X

X_aligned = X.reindex(columns=schema_loaded, fill_value=0)

if list(X_aligned.columns) != list(schema_loaded):
    raise ValueError(
        "El orden de columnas de X_aligned no coincide con schema_loaded. "
        "Esto puede romper la inferencia porque LightGBM depende del orden de columnas."
    )

print("Features transformadas y alineadas:")
print(f" - X original    : {X.shape}")
print(f" - X_aligned     : {X_aligned.shape}")
print(f" - schema_loaded : {len(schema_loaded)} features")

# ------------------------------------------------------------
# 6. Forzar regeneración de scores con model_5tx_old.pkl
# ------------------------------------------------------------

print("\nRegenerando scores usando model_5tx_old.pkl...")

X_np = X_aligned.values.astype("float32")
proba = model_loaded.predict_proba(X_np)[:, 1]

print("Scores generados:")
print(f" - n_scores   : {len(proba):,}")
print(f" - min_score  : {proba.min():.8f}")
print(f" - mean_score : {proba.mean():.8f}")
print(f" - max_score  : {proba.max():.8f}")

# ------------------------------------------------------------
# 7. Checks duros de consistencia
# ------------------------------------------------------------

if len(data_scoring) != len(proba):
    raise ValueError(
        f"Inconsistencia de filas: data_scoring={len(data_scoring):,}, "
        f"proba={len(proba):,}."
    )

if len(data_scoring) != len(X_aligned):
    raise ValueError(
        f"Inconsistencia de filas: data_scoring={len(data_scoring):,}, "
        f"X_aligned={len(X_aligned):,}."
    )

if hasattr(model_loaded, "n_features_in_"):
    if model_loaded.n_features_in_ != X_aligned.shape[1]:
        raise ValueError(
            f"Inconsistencia de features: modelo espera {model_loaded.n_features_in_}, "
            f"pero X_aligned tiene {X_aligned.shape[1]}."
        )

print("\nChecks OK:")
print(f" - Filas originales scoreadas : {len(data_scoring):,}")
print(f" - Features modelo internas   : {X_aligned.shape[1]:,}")
print(f" - Scores generados           : {len(proba):,}")
print(f" - Modelo usado               : {model_artifact_name}")

# ------------------------------------------------------------
# 8. Crear tabla final: tabla original + score + metadata modelo
# ------------------------------------------------------------

df_model_scoring_raw = data_scoring.reset_index(drop=True).copy()

cols_to_drop_if_exist = [
    "model_row_id",
    "score_generated_ts",
    "p_5tx_30d",
    "score_rank_desc",
    "score_decile",
    "model_artifact_name",
    "model_artifact_path",
    "model_artifact_sha256"
]

df_model_scoring_raw = df_model_scoring_raw.drop(
    columns=[c for c in cols_to_drop_if_exist if c in df_model_scoring_raw.columns],
    errors="ignore"
)

tz_local = (
    getattr(cfg_loaded, "tz_local", "America/Mexico_City")
    if "cfg_loaded" in globals() and cfg_loaded is not None
    else "America/Mexico_City"
)

score_ts = pd.Timestamp.now(tz=tz_local)

df_model_scoring_raw.insert(
    0,
    "model_row_id",
    np.arange(len(df_model_scoring_raw), dtype=np.int64)
)

df_model_scoring_raw.insert(
    1,
    "score_generated_ts",
    score_ts
)

df_model_scoring_raw["p_5tx_30d"] = proba.astype("float32")

df_model_scoring_raw["model_artifact_name"] = model_artifact_name
df_model_scoring_raw["model_artifact_path"] = model_artifact_path
df_model_scoring_raw["model_artifact_sha256"] = model_artifact_sha256

df_model_scoring_raw["score_rank_desc"] = (
    df_model_scoring_raw["p_5tx_30d"]
    .rank(method="first", ascending=False)
    .astype("int64")
)

df_model_scoring_raw["score_decile"] = pd.qcut(
    df_model_scoring_raw["p_5tx_30d"].rank(method="first"),
    q=10,
    labels=False,
    duplicates="drop"
).astype("int16") + 1

df_model_scoring_raw["score_decile"] = 11 - df_model_scoring_raw["score_decile"]

print("\nTabla raw + score generada:")
print(f" - Shape: {df_model_scoring_raw.shape}")
print(f" - Columnas originales conservadas: {data_scoring.shape[1]}")
print(f" - Modelo usado: {model_artifact_name}")

display(df_model_scoring_raw.head())

# ------------------------------------------------------------
# 9. Preparar DataFrame BigQuery-safe
# ------------------------------------------------------------

df_bq = df_model_scoring_raw.copy()

# Limpiar nombres de columnas
clean_cols = [clean_bq_colname(c) for c in df_bq.columns]
df_bq.columns = make_unique_columns(clean_cols)

if df_bq.columns.duplicated().any():
    duplicated = df_bq.columns[df_bq.columns.duplicated()].tolist()
    raise ValueError(f"Columnas duplicadas después de limpiar nombres: {duplicated}")

# ------------------------------------------------------------
# 10. Convertir casi todo a STRING
#     Mantener numéricos clave como numéricos.
# ------------------------------------------------------------

numeric_keep_cols = [
    "model_row_id",
    "p_5tx_30d",
    "score_rank_desc",
    "score_decile"
]

numeric_keep_cols = [clean_bq_colname(c) for c in numeric_keep_cols]
numeric_keep_cols = [c for c in numeric_keep_cols if c in df_bq.columns]

# Convertir todo excepto columnas numéricas clave a string
for col in df_bq.columns:
    if col not in numeric_keep_cols:
        df_bq[col] = (
            df_bq[col]
            .astype("string")
            .str.replace("\r", " ", regex=False)
            .str.replace("\n", " ", regex=False)
            .str.replace("\t", " ", regex=False)
            .str.replace("\x00", "", regex=False)
        )

# Reforzar tipos numéricos clave
if "model_row_id" in df_bq.columns:
    df_bq["model_row_id"] = pd.to_numeric(
        df_bq["model_row_id"],
        errors="coerce"
    ).astype("Int64")

if "p_5tx_30d" in df_bq.columns:
    df_bq["p_5tx_30d"] = pd.to_numeric(
        df_bq["p_5tx_30d"],
        errors="coerce"
    ).astype("float64")

if "score_rank_desc" in df_bq.columns:
    df_bq["score_rank_desc"] = pd.to_numeric(
        df_bq["score_rank_desc"],
        errors="coerce"
    ).astype("Int64")

if "score_decile" in df_bq.columns:
    df_bq["score_decile"] = pd.to_numeric(
        df_bq["score_decile"],
        errors="coerce"
    ).astype("Int64")

# Reemplazar infinitos por nulos en columnas numéricas
for col in numeric_keep_cols:
    if col in df_bq.columns:
        df_bq[col] = df_bq[col].replace([np.inf, -np.inf], np.nan)

print("\nDataFrame BigQuery-safe listo:")
print(f" - Shape final: {df_bq.shape}")
print(f" - Columnas duplicadas: {df_bq.columns.duplicated().sum()}")
print(f" - Columnas numéricas conservadas: {numeric_keep_cols}")

display(df_bq.head())

# ------------------------------------------------------------
# 11. Cargar directo a BigQuery
# ------------------------------------------------------------

client = bigquery.Client(project=PROJECT_ID)

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True
)

job = client.load_table_from_dataframe(
    df_bq,
    TABLE_ID,
    job_config=job_config
)

job.result()

table = client.get_table(TABLE_ID)

print("\nCarga robusta completada en BigQuery")
print(f"Tabla destino: {TABLE_ID}")
print(f"Filas cargadas: {table.num_rows:,}")
print(f"Columnas cargadas: {len(table.schema):,}")

# ------------------------------------------------------------
# 12. Auditoría local del artifact usado y carga
# ------------------------------------------------------------

df_artifact_audit = pd.DataFrame([{
    "table_id": TABLE_ID,
    "model_artifact_name": model_artifact_name,
    "model_artifact_path": model_artifact_path,
    "model_artifact_sha256": model_artifact_sha256,
    "n_rows_scored": len(df_model_scoring_raw),
    "n_rows_loaded_bq": table.num_rows,
    "n_model_features": X_aligned.shape[1],
    "score_min": float(np.min(proba)),
    "score_mean": float(np.mean(proba)),
    "score_max": float(np.max(proba)),
    "score_generated_ts": str(score_ts)
}])

display(df_artifact_audit)

OUTPUT_DIR = Path("mlops_artifacts_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

artifact_audit_path = OUTPUT_DIR / "artifact_audit_bq_load_OLD_MODEL.csv"
df_artifact_audit.to_csv(artifact_audit_path, index=False)

print(f"\nAuditoría guardada en: {artifact_audit_path}")

Modelo activo cargado:
 - artifact_name : model_5tx_old.pkl
 - artifact_path : mlops_artifacts_v2/model_5tx_old.pkl
 - sha256        : c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8cc1a47573c45bf64f9
 - object_type   : <class 'lightgbm.sklearn.LGBMClassifier'>

Regenerando features transformadas desde data_scoring...
Features transformadas y alineadas:
 - X original    : (2371692, 64)
 - X_aligned     : (2371692, 64)
 - schema_loaded : 64 features

Regenerando scores usando model_5tx_old.pkl...
Scores generados:
 - n_scores   : 2,371,692
 - min_score  : 0.00288018
 - mean_score : 0.66416867
 - max_score  : 0.99982221

Checks OK:
 - Filas originales scoreadas : 2,371,692
 - Features modelo internas   : 64
 - Scores generados           : 2,371,692
 - Modelo usado               : model_5tx_old.pkl

Tabla raw + score generada:
 - Shape: (2371692, 45)
 - Columnas originales conservadas: 37
 - Modelo usado: model_5tx_old.pkl


,model_row_id,score_generated_ts,user_id,signup_date,signup_ts,userTypeIdentifier,channelUserIdentifier,accountLevel,stateName,gender,user_type,channelDetail,birth_date,birthState,Card_linked_date,IsActive,phn_confir,email_confir,phone_conf_ts,email_conf_ts,phn_confir_d7,email_confir_d7,both_confir_d7,premia_accountid,has_premia,activation_date_ever,activation_date_30d,label_activated_30d,tx_30d_count,tx_30d_amount,label_5tx_30d,first_tx_type,first_tx_amount,activation_channel,latest_tx_date,lifespan_days,days_since_last,tx_30d_from_activation,days_to_first_activation,p_5tx_30d,model_artifact_name,model_artifact_path,model_artifact_sha256,score_rank_desc,score_decile
0,0,2026-04-30 15:25:46.083116-06:00,0005eaa7-4309-4a9d-a43d-7d01a7849220,2025-12-07,2025-12-07 20:16:48.643000+00:00,3,1,2,NL,male,HYBRID,POS,1980-08-17,NL,2025-12-07,True,1,0,2025-12-08 02:32:18.850000+00:00,NaT,1,0,0,2WHVMJB,1,2025-12-07,2025-12-07,1,55,61742.0000,1,SPEI_CASH_IN,2.0000,SPEI/Transfer,2026-04-09,124,1,55,0,0.9977,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,150615,1
1,1,2026-04-30 15:25:46.083116-06:00,00073792-5c60-4ab1-a712-719f260fac54,2025-09-26,2025-09-26 19:47:33.124000+00:00,3,1,2,NL,female,HYBRID,POS,1989-08-11,VE,2025-09-26,True,1,1,2025-09-27 03:06:57.839000+00:00,2025-09-27 03:03:41.652000+00:00,1,1,1,7F5W5LZ,1,2025-09-27,2025-09-27,1,58,29216.3500,1,SPEI_CASH_IN,3960.3000,SPEI/Transfer,2026-03-29,184,12,61,1,0.9977,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,150998,1
2,2,2026-04-30 15:25:46.083116-06:00,000bb768-1e8d-4440-b134-215ff1eaf51a,2025-11-25,2025-11-26 02:30:29.093000+00:00,3,1,3,PU,male,HYBRID,POS,1999-04-29,OA,2025-11-25,True,1,1,2025-11-26 08:41:35.788000+00:00,2025-11-26 08:40:45.193000+00:00,1,1,1,JPTOCIW,1,2025-11-27,2025-11-27,1,42,160914.0000,1,SPEI_CASH_IN,1800.0000,SPEI/Transfer,2026-04-04,129,6,44,2,0.9992,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,25149,1
3,3,2026-04-30 15:25:46.083116-06:00,000d8f79-8094-4489-a6d4-5c409199014e,2025-02-05,2025-02-06 03:13:05.792000+00:00,3,1,3,NL,female,HYBRID,POS,2001-03-02,NL,2025-02-05,True,1,1,2025-02-06 09:36:36.453000+00:00,2025-03-06 08:55:57.868000+00:00,1,0,0,UCHGI6G,1,2025-02-05,2025-02-05,1,52,18920.7600,1,SPEI_CASH_IN,0.9400,SPEI/Transfer,2026-04-08,428,2,52,0,0.9942,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,322148,2
4,4,2026-04-30 15:25:46.083116-06:00,000f24ac-dafc-40ad-9032-7fcd70f77d4d,2025-12-04,2025-12-04 22:42:55.637000+00:00,3,1,2,SL,male,HYBRID,POS,1978-06-15,SL,2025-12-04,True,1,1,2025-12-05 05:02:51.935000+00:00,2026-01-09 13:12:17.566000+00:00,1,0,0,CDHMAUK,1,2025-12-10,2025-12-10,1,33,5975.2900,1,CASH_IN_AT_OXXO_QR,200.0000,QR,2026-04-08,120,2,42,6,0.9465,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,863713,4



DataFrame BigQuery-safe listo:
 - Shape final: (2371692, 45)
 - Columnas duplicadas: 0
 - Columnas numéricas conservadas: ['model_row_id', 'p_5tx_30d', 'score_rank_desc', 'score_decile']


,model_row_id,score_generated_ts,user_id,signup_date,signup_ts,usertypeidentifier,channeluseridentifier,accountlevel,statename,gender,user_type,channeldetail,birth_date,birthstate,card_linked_date,isactive,phn_confir,email_confir,phone_conf_ts,email_conf_ts,phn_confir_d7,email_confir_d7,both_confir_d7,premia_accountid,has_premia,activation_date_ever,activation_date_30d,label_activated_30d,tx_30d_count,tx_30d_amount,label_5tx_30d,first_tx_type,first_tx_amount,activation_channel,latest_tx_date,lifespan_days,days_since_last,tx_30d_from_activation,days_to_first_activation,p_5tx_30d,model_artifact_name,model_artifact_path,model_artifact_sha256,score_rank_desc,score_decile
0,0,2026-04-30 15:25:46.083116-06:00,0005eaa7-4309-4a9d-a43d-7d01a7849220,2025-12-07,2025-12-07 20:16:48.643000+00:00,3,1,2,NL,male,HYBRID,POS,1980-08-17,NL,2025-12-07,True,1,0,2025-12-08 02:32:18.850000+00:00,<NA>,1,0,0,2WHVMJB,1,2025-12-07,2025-12-07,1,55,61742.0,1,SPEI_CASH_IN,2.0,SPEI/Transfer,2026-04-09,124,1,55,0,0.9977,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,150615,1
1,1,2026-04-30 15:25:46.083116-06:00,00073792-5c60-4ab1-a712-719f260fac54,2025-09-26,2025-09-26 19:47:33.124000+00:00,3,1,2,NL,female,HYBRID,POS,1989-08-11,VE,2025-09-26,True,1,1,2025-09-27 03:06:57.839000+00:00,2025-09-27 03:03:41.652000+00:00,1,1,1,7F5W5LZ,1,2025-09-27,2025-09-27,1,58,29216.350000000002,1,SPEI_CASH_IN,3960.3,SPEI/Transfer,2026-03-29,184,12,61,1,0.9977,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,150998,1
2,2,2026-04-30 15:25:46.083116-06:00,000bb768-1e8d-4440-b134-215ff1eaf51a,2025-11-25,2025-11-26 02:30:29.093000+00:00,3,1,3,PU,male,HYBRID,POS,1999-04-29,OA,2025-11-25,True,1,1,2025-11-26 08:41:35.788000+00:00,2025-11-26 08:40:45.193000+00:00,1,1,1,JPTOCIW,1,2025-11-27,2025-11-27,1,42,160914.0,1,SPEI_CASH_IN,1800.0,SPEI/Transfer,2026-04-04,129,6,44,2,0.9992,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,25149,1
3,3,2026-04-30 15:25:46.083116-06:00,000d8f79-8094-4489-a6d4-5c409199014e,2025-02-05,2025-02-06 03:13:05.792000+00:00,3,1,3,NL,female,HYBRID,POS,2001-03-02,NL,2025-02-05,True,1,1,2025-02-06 09:36:36.453000+00:00,2025-03-06 08:55:57.868000+00:00,1,0,0,UCHGI6G,1,2025-02-05,2025-02-05,1,52,18920.76,1,SPEI_CASH_IN,0.94,SPEI/Transfer,2026-04-08,428,2,52,0,0.9942,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,322148,2
4,4,2026-04-30 15:25:46.083116-06:00,000f24ac-dafc-40ad-9032-7fcd70f77d4d,2025-12-04,2025-12-04 22:42:55.637000+00:00,3,1,2,SL,male,HYBRID,POS,1978-06-15,SL,2025-12-04,True,1,1,2025-12-05 05:02:51.935000+00:00,2026-01-09 13:12:17.566000+00:00,1,0,0,CDHMAUK,1,2025-12-10,2025-12-10,1,33,5975.29,1,CASH_IN_AT_OXXO_QR,200.0,QR,2026-04-08,120,2,42,6,0.9465,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,863713,4



Carga robusta completada en BigQuery
Tabla destino: spin-aip-singularity-data-sb.Test_predictions_MLOps_30D_model.tbl_model_5trx_scoring_mlop_test_v1
Filas cargadas: 2,371,692
Columnas cargadas: 45


,table_id,model_artifact_name,model_artifact_path,model_artifact_sha256,n_rows_scored,n_rows_loaded_bq,n_model_features,score_min,score_mean,score_max,score_generated_ts
0,spin-aip-singularity-data-sb.Test_predictions_...,model_5tx_old.pkl,mlops_artifacts_v2/model_5tx_old.pkl,c1a24f7e85260599adce096e42e47ae831d2a78c23e2a8...,2371692,2371692,64,0.0029,0.6642,0.9998,2026-04-30 15:25:46.083116-06:00



Auditoría guardada en: mlops_artifacts_v2/artifact_audit_bq_load_OLD_MODEL.csv
